In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys
sys.dont_write_bytecode = True
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(variable, '1')
NOTEBOOK_NAME = '07_01_first_level_glm_and_group_inference.ipynb'
NOTEBOOK_RELATIVE_PATH = '07_01_first_level_glm_and_group_inference.ipynb'
WORKING_DIRECTORY = Path.cwd().resolve()
NOTEBOOK_CANDIDATES = [Path(os.environ['A8_NOTEBOOK_PATH']).expanduser()] if 'A8_NOTEBOOK_PATH' in os.environ else [WORKING_DIRECTORY / NOTEBOOK_NAME, WORKING_DIRECTORY / NOTEBOOK_RELATIVE_PATH]
NOTEBOOK_PATH = next((p.resolve() for p in NOTEBOOK_CANDIDATES if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError('Set A8_NOTEBOOK_PATH to this notebook, or run with its containing directory as the working directory')
ANALYSIS_ROOT = NOTEBOOK_PATH.parent


In [ ]:
from pathlib import Path as u12_Path
import sys as u12_sys
from IPython.display import display as u12_display, Image as u12_Image
import argparse as u12_argparse
import hashlib as u12_hashlib
import json as u12_json
import os as u12_os
import sys as u12_sys
import traceback as u12_traceback
from pathlib import Path as u12_Path
import nibabel as u12_nib
import numpy as u12_np
import pandas as u12_pd
from nilearn.glm.first_level import FirstLevelModel as u12_FirstLevelModel
from nilearn.glm.contrasts import compute_fixed_effects as u12__compute_fixed_effects


In [ ]:
def u12_glm_compute_fixed_effects(*args, **kwargs):
    kwargs.pop('return_z_score', None)
    result = u12__compute_fixed_effects(*args, **kwargs)
    return tuple((u12_nib.Nifti1Image(u12_np.asarray(img.dataobj, dtype=u12_np.float32), img.affine) for img in result))


In [ ]:
from nilearn.masking import intersect_masks as u12_intersect_masks
import sys as u12_sys
from pathlib import Path as u12_Path
u12_glm_TR = 2.0
u12_glm_SMOOTH_FWHM = 6.0
u12_glm_HIGH_PASS_HZ = 1.0 / 128.0
u12_glm_FD_THRESHOLD = 0.5
u12_glm_STDDVARS_THRESHOLD = 1.5
u12_glm_STROOP_DURATION_S = 1.0
u12_glm_MOTION_BASE = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']
u12_glm_MODEL_VERSIONS = {'stroop': 'public-observed-events-glover-v1-stroop-duration1s', 'gonogo': 'public-observed-events-glover-v1-gonogo-duration500ms'}


In [ ]:
def u12_glm_one(path_pattern: str) -> u12_Path:
    hits = sorted(u12_Path().glob(path_pattern)) if not path_pattern.startswith('/') else []
    if path_pattern.startswith('/'):
        import glob as glob
        hits = [u12_Path(x) for x in sorted(glob.glob(path_pattern))]
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected exactly one match, found {len(hits)}: {path_pattern}')
    return hits[0]


In [ ]:
def u12_glm_paths_for(subject, task, run=None):
    return u12_imaging_paths(subject, task, run)


In [ ]:
def u12_glm_load_events(path: u12_Path, task: str) -> u12_pd.DataFrame:
    ev = u12_pd.read_csv(path, sep=',' if u12_Path(path).suffix.lower() == '.csv' else '\t', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    required = {'onset', 'duration', 'trial_type'}
    if not required.issubset(ev.columns):
        raise ValueError(f'Missing event columns {required - set(ev.columns)} in {path}')
    ev = ev[['onset', 'duration', 'trial_type'] + (['accuracy'] if 'accuracy' in ev.columns else [])].copy()
    if task == 'stroop':
        if 'accuracy' not in ev.columns:
            raise ValueError(f'Stroop accuracy column absent: {path}')
        accuracy = u12_pd.to_numeric(ev['accuracy'], errors='raise')
        if not accuracy.isin([0, 1]).all():
            raise ValueError('Stroop event accuracy must be observed 0/1')
        correct = accuracy.eq(1)
        ev.loc[~correct, 'trial_type'] = 'error'
    elif 'accuracy' in ev.columns and set(ev['trial_type'].astype(str).str.lower()).issubset({'go', 'nogo'}):
        accuracy = u12_pd.to_numeric(ev['accuracy'], errors='raise')
        if not accuracy.isin([0, 1]).all():
            raise ValueError('Go/No-Go event accuracy must be observed 0/1')
        acc = accuracy.eq(1)
        base = ev['trial_type'].astype(str).str.lower()
        ev['trial_type'] = u12_np.where(acc, base + '_correct', base + '_error')
    if 'accuracy' in ev.columns:
        ev = ev.drop(columns='accuracy')
    ev['trial_type'] = ev['trial_type'].astype(str).str.lower()
    ev['onset'] = u12_pd.to_numeric(ev['onset'], errors='raise')
    ev['duration'] = u12_pd.to_numeric(ev['duration'], errors='raise')
    allowed = {'congruent', 'neutral', 'incongruent', 'error'} if task == 'stroop' else {'go_correct', 'nogo_correct', 'go_error', 'nogo_error'}
    if not ev.trial_type.isin(allowed).all() or not u12_np.isfinite(ev[['onset', 'duration']].to_numpy(float)).all() or (ev.duration < 0).any():
        raise ValueError('Unknown event type or invalid event timing')
    if task == 'stroop':
        ev['duration'] = u12_glm_STROOP_DURATION_S
    if task == 'gonogo':
        ev['duration'] = 0.5
    return ev


In [ ]:
def u12_glm_raw_log_path(subject, task, run=None):
    return u12_glm_paths_for(subject, task, run)['events']


In [ ]:
def u12_glm_reconstruct_events(subject, task, run=None):
    return u12_glm_load_events(u12_glm_raw_log_path(subject, task, run), task)


In [ ]:
def u12_glm_signature_paths(subject, task):
    inputs = []
    for run in [None] if task == 'stroop' else ['01', '02']:
        paths = u12_glm_paths_for(subject, task, run)
        inputs.extend(paths.values())
    return inputs + [u12_Path(NOTEBOOK_PATH), u12_input_path('MRI_manifest.csv')]


In [ ]:
def u12_glm_input_signature(subject, task):
    payload = [(str(p), p.stat().st_size, p.stat().st_mtime_ns) for p in u12_glm_signature_paths(subject, task)]
    source_signature = u15_source_signature
    return u12_hashlib.sha256(u12_json.dumps([payload, source_signature()]).encode()).hexdigest()


In [ ]:
def u12_glm_load_confounds(path: u12_Path) -> tuple[u12_pd.DataFrame, dict[str, float | int]]:
    cf = u12_pd.read_csv(path, sep=',' if u12_Path(path).suffix.lower() == '.csv' else '\t', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    required = {'framewise_displacement', 'std_dvars'}
    if not required.issubset(cf.columns):
        raise ValueError(f'Required spike-screening inputs absent: {required - set(cf.columns)}')
    selected: dict[str, u12_pd.Series] = {}
    for base in u12_glm_MOTION_BASE:
        for col in (base, f'{base}_derivative1', f'{base}_power2', f'{base}_derivative1_power2'):
            if col not in cf.columns:
                raise ValueError(f'Required motion confound absent: {col} in {path}')
            selected[col] = u12_pd.to_numeric(cf[col], errors='coerce').fillna(0.0)
    for col in cf.columns:
        if col.startswith('non_steady_state_outlier'):
            selected[col] = u12_pd.to_numeric(cf[col], errors='coerce').fillna(0.0)
    fd = u12_pd.to_numeric(cf.get('framewise_displacement', u12_pd.Series(0.0, index=cf.index)), errors='coerce').fillna(0.0)
    dv = u12_pd.to_numeric(cf.get('std_dvars', u12_pd.Series(0.0, index=cf.index)), errors='coerce').fillna(0.0)
    if float((fd > u12_glm_FD_THRESHOLD).mean()) > 0.2:
        raise ValueError('Run fails manuscript motion inclusion: more than 20% volumes with FD > 0.5 mm; revise the participant cohort before fitting')
    censor = (fd > u12_glm_FD_THRESHOLD) | (dv > u12_glm_STDDVARS_THRESHOLD)
    for idx in u12_np.flatnonzero(censor.to_numpy()):
        spike = u12_np.zeros(len(cf), dtype=float)
        spike[idx] = 1.0
        selected[f'motion_outlier_{idx:04d}'] = u12_pd.Series(spike)
    out = u12_pd.DataFrame(selected, index=cf.index).astype(float)
    info = {'n_volumes': int(len(cf)), 'n_confounds': int(out.shape[1]), 'n_spikes': int(censor.sum()), 'spike_pct': float(censor.mean() * 100.0), 'mean_fd_mm': float(fd.mean()), 'max_fd_mm': float(fd.max()), 'mean_std_dvars': float(dv.mean())}
    return (out, info)


In [ ]:
def u12_glm_acquisition(path: u12_Path) -> tuple[float, float]:
    meta = u12_json.loads(path.read_text(encoding='utf-8'))
    tr = float(meta.get('RepetitionTime', u12_glm_TR))
    start = float(meta.get('StartTime', 0.0))
    if not u12_np.isclose(tr, u12_glm_TR):
        raise ValueError(f'Unexpected TR={tr}: {path}')
    return (tr, start / tr)


In [ ]:
def u12_glm_design_diagnostics(dm: u12_pd.DataFrame) -> dict[str, float | int | str]:
    x = dm.to_numpy(dtype=float)
    rank = int(u12_np.linalg.matrix_rank(x))
    variable = u12_np.std(x, axis=0) > u12_np.finfo(float).eps
    x_scaled = x[:, variable]
    x_scaled = (x_scaled - x_scaled.mean(axis=0)) / x_scaled.std(axis=0)
    singular = u12_np.linalg.svd(x_scaled, compute_uv=False)
    cond = float(singular[0] / singular[-1]) if singular[-1] > 0 else float('inf')
    return {'n_timepoints': int(x.shape[0]), 'n_regressors': int(x.shape[1]), 'design_rank': rank, 'rank_deficient': int(rank < x.shape[1]), 'condition_number': cond, 'columns': '|'.join(map(str, dm.columns))}


In [ ]:
def u12_glm_fit_task(subject: str, task: str) -> list[dict]:
    runs = [None] if task == 'stroop' else ['01', '02']
    pp = [u12_glm_paths_for(subject, task, r) for r in runs]
    for p in pp:
        for (key, value) in p.items():
            if key == 'events':
                continue
            if not value.exists():
                raise FileNotFoundError(value)
    (trs, refs) = zip(*(u12_glm_acquisition(p['metadata']) for p in pp))
    imgs = [str(p['bold']) for p in pp]
    masks = []
    for p in pp:
        original = u12_nib.load(p['mask'])
        brain = u12_np.asarray(original.dataobj) > 0
        data = u12_np.asarray(u12_nib.load(p['bold']).dataobj, dtype='float32')
        series = data[brain]
        valid = brain.copy()
        valid[brain] = u12_np.isfinite(series).all(1) & (series.std(1) > 0) & (abs(series.mean(1)) > 1e-06)
        masks.append(u12_nib.Nifti1Image(valid.astype('uint8'), original.affine))
        del data, series
    mask_img = masks[0] if len(masks) == 1 else u12_intersect_masks(masks, threshold=1.0, connected=False)
    events = []
    event_sources = []
    for p in pp:
        if task == 'stroop':
            events.append(u12_glm_reconstruct_events(subject, task, None))
            event_sources.append('observed_events_TSV')
        elif p['events'].exists():
            events.append(u12_glm_load_events(p['events'], task))
            event_sources.append('BIDS_events_uniform')
        else:
            raise FileNotFoundError(f"BIDS events absent: {p['events']}")
    (confounds, infos) = zip(*(u12_glm_load_confounds(p['confounds']) for p in pp))
    for (img_path, cf, info) in zip(imgs, confounds, infos):
        n_img = int(u12_nib.load(img_path).shape[-1])
        if n_img != len(cf):
            raise ValueError(f'Volume/confound mismatch ({n_img} vs {len(cf)}): {img_path}')
    if task == 'stroop':
        contrast_name = 'incongruent_minus_congruent'
        contrast = 'incongruent - congruent'
    else:
        contrast_name = 'nogo_correct_minus_go_correct'
        contrast = 'nogo_correct - go_correct'
    models = []
    run_effects = []
    run_variances = []
    run_zscores = []
    dofs = []
    event_counts = []
    for (run, img, ev, cf, tr, ref) in zip(runs, imgs, events, confounds, trs, refs):
        model = u12_FirstLevelModel(t_r=float(tr), slice_time_ref=float(ref), hrf_model='glover + derivative', drift_model='cosine', high_pass=u12_glm_HIGH_PASS_HZ, noise_model='ar1', smoothing_fwhm=u12_glm_SMOOTH_FWHM, mask_img=mask_img, signal_scaling=0, minimize_memory=True, n_jobs=1, verbose=0)
        model = model.fit(img, events=ev, confounds=cf)
        event_counts.append(ev['trial_type'].value_counts().astype(int).to_dict())
        models.append(model)
        run_effects.append(model.compute_contrast(contrast, output_type='effect_size'))
        run_variances.append(model.compute_contrast(contrast, output_type='effect_variance'))
        run_zscores.append(model.compute_contrast(contrast, output_type='z_score'))
        dm = model.design_matrices_[0]
        dofs.append(int(dm.shape[0] - u12_np.linalg.matrix_rank(dm.to_numpy(dtype=float))))
    if len(runs) == 1:
        final_maps = {'effect': run_effects[0], 'variance': run_variances[0], 'z': run_zscores[0]}
    else:
        (effect, variance, _stat, zscore) = u12_glm_compute_fixed_effects(run_effects, run_variances, mask=mask_img, precision_weighted=True, dofs=dofs, return_z_score=True)
        final_maps = {'effect': effect, 'variance': variance, 'z': zscore}
    final_maps = {key: u12_nib.Nifti1Image(u12_np.asarray(img.dataobj, dtype=u12_np.float32), img.affine) for (key, img) in final_maps.items()}
    out_dir = u12_glm_OUT_ROOT
    out_dir.mkdir(parents=True, exist_ok=True)
    variance = u12_np.asarray(final_maps['variance'].dataobj)
    valid_model = u12_np.asarray(mask_img.dataobj, dtype=bool) & u12_np.isfinite(variance) & (variance > 0)
    valid_model &= u12_np.isfinite(u12_np.asarray(final_maps['effect'].dataobj))
    u12_nib.save(u12_nib.Nifti1Image(valid_model.astype('uint8'), final_maps['variance'].affine), out_dir / f'{subject}_task-{task}_mask-analysis.nii.gz')
    for (suffix, img) in final_maps.items():
        img = u12_nib.Nifti1Image(u12_np.where(valid_model, u12_np.asarray(img.dataobj), 0).astype('float32'), img.affine)
        u12_nib.save(img, out_dir / f'{subject}_task-{task}_contrast-{contrast_name}_stat-{suffix}.nii.gz')
    rows = []
    for (run, model, info, counts, source, ref) in zip(runs, models, infos, event_counts, event_sources, refs):
        dm = model.design_matrices_[0]
        row = {'subject_id': subject, 'task': task, 'run': run or 'single', 'model_version': u12_glm_MODEL_VERSIONS[task], 'slice_time_ref': ref, 'event_source': source}
        row.update(info)
        row.update(u12_glm_design_diagnostics(dm))
        for (condition, n) in counts.items():
            row[f'trials_{condition}'] = int(n)
        rows.append(row)
    return rows


In [ ]:
def u12_glm_expected_outputs(subject: str, tasks: list[str] | None=None) -> list[u12_Path]:
    out = u12_glm_OUT_ROOT
    names = [('stroop', 'incongruent_minus_congruent'), ('gonogo', 'nogo_correct_minus_go_correct')]
    selected = set(tasks or ('stroop', 'gonogo'))
    return [out / f'{subject}_task-{task}_contrast-{contrast}_stat-{stat}.nii.gz' for (task, contrast) in names if task in selected for stat in ('effect', 'variance', 'z')] + [out / f'{subject}_task-{task}_mask-analysis.nii.gz' for task in selected]


In [ ]:
def u12_glm_process_subject(subject: str, force: bool=False, tasks: list[str] | None=None) -> int:
    tasks = tasks or ['stroop', 'gonogo']
    if set(tasks) == {'stroop', 'gonogo'}:
        return int(any([u12_glm_process_subject(subject, force=force, tasks=[task]) for task in tasks]))
    u12_glm_QC_ROOT.mkdir(parents=True, exist_ok=True)
    if set(tasks) not in ({'stroop'}, {'gonogo'}):
        raise ValueError('Write one task at a time to keep flattened QC files distinct')
    qc_label = 'stroop' if tasks == ['stroop'] else 'gonogo_duration500ms'
    qc_file = u12_glm_QC_ROOT / f'{subject}_task-{qc_label}_first_level_qc.csv'
    old_qc = u12_pd.read_csv(qc_file, encoding='utf-8-sig', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}) if qc_file.exists() else u12_pd.DataFrame()
    signature = u12_glm_input_signature(subject, tasks[0])
    selected_qc_current = False
    if not old_qc.empty and {'task', 'model_version', 'event_source', 'rank_deficient'}.issubset(old_qc.columns):
        selected_qc_current = not old_qc.loc[old_qc.task.isin(tasks), 'rank_deficient'].astype(bool).any() and all((len(old_qc.loc[old_qc.task.eq(task)]) == (1 if task == 'stroop' else 2) and old_qc.loc[old_qc.task.eq(task), 'model_version'].eq(u12_glm_MODEL_VERSIONS[task]).all() and old_qc.loc[old_qc.task.eq(task), 'event_source'].eq('observed_events_TSV' if task == 'stroop' else 'BIDS_events_uniform').all() for task in tasks))
    if not force and selected_qc_current and ('input_signature' in old_qc) and old_qc.input_signature.eq(signature).all() and all((p.exists() for p in u12_glm_expected_outputs(subject, tasks))):
        print(f'SKIP {subject}: selected tasks complete', flush=True)
        return 0
    try:
        qc_file.unlink(missing_ok=True)
        rows = []
        for task in tasks:
            rows.extend(u12_glm_fit_task(subject, task))
        new_qc = u12_pd.DataFrame(rows)
        new_qc['input_signature'] = signature
        if not old_qc.empty:
            old_qc = old_qc.loc[~old_qc.task.isin(tasks)]
            new_qc = u12_pd.concat([old_qc, new_qc], ignore_index=True, sort=False)
        new_qc = new_qc.sort_values(['task', 'run']).reset_index(drop=True)
        temporary = qc_file.with_suffix('.tmp')
        new_qc.to_csv(temporary, index=False, encoding='utf-8-sig')
        temporary.replace(qc_file)
        print(f'OK {subject}', flush=True)
        return 0
    except Exception:
        print(f'FAIL {subject}\n{u12_traceback.format_exc()}', file=u12_sys.stderr, flush=True)
        return 1


In [ ]:
def u12_glm_subject_list():
    return u12_participants().subject_id.tolist()


In [ ]:
def u12_glm_main() -> int:
    parser = u12_argparse.ArgumentParser()
    parser.add_argument('--subject', action='append', help='Subject such as sub-9001; repeatable. Default: all 106.')
    parser.add_argument('--task', action='append', choices=('stroop', 'gonogo'), help='Task to fit; repeatable. Default: both tasks.')
    parser.add_argument('--force', action='store_true')
    args = parser.parse_args()
    subjects = args.subject or u12_glm_subject_list()
    unknown = sorted(set(subjects) - set(u12_glm_subject_list()))
    if unknown:
        parser.error(f'Subjects outside locked 106-person set: {unknown}')
    tasks = list(dict.fromkeys(args.task or ['stroop', 'gonogo']))
    failures = [u12_glm_process_subject(s, force=args.force, tasks=[task]) for s in subjects for task in tasks]
    return int(any(failures))


In [ ]:
import json as u12_json
import math as u12_math
import os as u12_os
import sys as u12_sys
from pathlib import Path as u12_Path
import matplotlib as u12_matplotlib
import matplotlib.pyplot as u12_plt
from matplotlib import font_manager as u12_font_manager
import nibabel as u12_nib
import nilearn as u12_nilearn
import numpy as u12_np
import pandas as u12_pd
from nilearn import datasets as u12_datasets, image as u12_image
from nilearn.glm.second_level import SecondLevelModel as u12_SecondLevelModel, non_parametric_inference as u12_non_parametric_inference
from nilearn.image import resample_to_img as u12_resample_to_img
from nilearn.masking import intersect_masks as u12_intersect_masks
from scipy import ndimage as u12_ndimage
u12_group_MNI_BACKGROUND = None
u12_MODEL_VERSIONS = u12_glm_MODEL_VERSIONS
u12_group_N_PERM = 5000
u12_group_RANDOM_STATE = 20260824
u12_group_FWE_ALPHA = 0.05
u12_group_CLUSTER_FORMING_P = 0.001
u12_group_TASKS = {'stroop': {'contrast': 'incongruent_minus_congruent', 'title': 'Stroop: Incongruent > Congruent'}, 'gonogo': {'contrast': 'nogo_correct_minus_go_correct', 'title': 'Go/No-Go: NoGo correct > Go correct'}}


In [ ]:
def u12_group_save_float(img, path):
    data = u12_np.asarray(img.dataobj)
    if not str(path).endswith('_mask.nii.gz'):
        data = data.astype(u12_np.float32)
    u12_nib.save(u12_nib.Nifti1Image(data, img.affine), path)


In [ ]:
def u12_group_effect_path(subject: str, task: str) -> u12_Path:
    contrast = u12_group_TASKS[task]['contrast']
    return u12_OUT_ROOT / f'{subject}_task-{task}_contrast-{contrast}_stat-effect.nii.gz'


In [ ]:
def u12_group_group_input_signatures(subjects, task):
    paths = [p for sid in subjects for p in [u12_group_effect_path(sid, task), u12_OUT_ROOT / f'{sid}_task-{task}_mask-analysis.nii.gz']]
    return [[str(p), p.stat().st_size, p.stat().st_mtime_ns] for p in paths]


In [ ]:
def u12_group_group_mask(subjects: list[str], task: str) -> u12_nib.Nifti1Image:
    masks = [str(u12_OUT_ROOT / f'{s}_task-{task}_mask-analysis.nii.gz') for s in subjects]
    return u12_intersect_masks(masks, threshold=1.0, connected=True)


In [ ]:
def u12_group_analysis_mask(subjects: list[str], task: str, name: str) -> u12_nib.Nifti1Image:
    stem = f'task-{task}_analysis-{name}'
    mask_path = u12_group_GROUP_ROOT / f'{stem}_mask.nii.gz'
    meta_path = u12_group_GROUP_ROOT / f'{stem}_metadata.json'
    if mask_path.exists() and meta_path.exists():
        meta = u12_json.loads(meta_path.read_text(encoding='utf-8'))
        signatures = u12_group_group_input_signatures(subjects, task)
        if meta.get('subjects') == subjects and meta.get('input_signatures') == signatures and (meta.get('first_level_model_version') == u12_MODEL_VERSIONS[task]) and (meta.get('code_sha256') == u12_source_signature()):
            return u12_nib.load(mask_path)
    return u12_group_group_mask(subjects, task)


In [ ]:
def u12_group_infer(name: str, subjects: list[str], task: str, cluster_inference: bool=False) -> dict:
    stem = f'task-{task}_analysis-{name}'
    maps = [str(u12_group_effect_path(s, task)) for s in subjects]
    missing = [p for p in maps if not u12_Path(p).exists()]
    if missing:
        raise FileNotFoundError(f'Missing {len(missing)} effect maps for {name}')
    input_signatures = u12_group_group_input_signatures(subjects, task)
    design = u12_pd.DataFrame({'intercept': u12_np.ones(len(subjects))})
    meta_path = u12_group_GROUP_ROOT / f'{stem}_metadata.json'
    required = [u12_group_GROUP_ROOT / f'{stem}_mask.nii.gz', u12_group_GROUP_ROOT / f'{stem}_stat-effect.nii.gz', u12_group_GROUP_ROOT / f'{stem}_stat-z.nii.gz', u12_group_GROUP_ROOT / f'{stem}_stat-t.nii.gz', u12_group_GROUP_ROOT / f'{stem}_stat-logpFWEmaxT.nii.gz']
    if cluster_inference:
        required.extend([u12_group_GROUP_ROOT / f'{stem}_stat-clusterSize_p001.nii.gz', u12_group_GROUP_ROOT / f'{stem}_stat-logpFWEclusterSize_p001.nii.gz'])
    if meta_path.exists() and all((p.exists() for p in required)):
        meta = u12_json.loads(meta_path.read_text(encoding='utf-8'))
        current = meta.get('code_sha256') == u12_source_signature() and meta.get('subjects') == subjects and (meta.get('first_level_model_version') == u12_MODEL_VERSIONS[task]) and (meta.get('input_signatures') == input_signatures) and (meta.get('nilearn_version') == u12_nilearn.__version__) and (meta.get('design_columns') == list(design.columns)) and (int(meta.get('n_permutations', -1)) == u12_group_N_PERM) and (bool(meta.get('cluster_inference', False)) == bool(cluster_inference))
        if current:
            return {'name': name, 'task': task, 'subjects': subjects, 'z': u12_nib.load(u12_group_GROUP_ROOT / f'{stem}_stat-z.nii.gz'), 'logp': u12_nib.load(u12_group_GROUP_ROOT / f'{stem}_stat-logpFWEmaxT.nii.gz'), 'cluster_logp': u12_nib.load(u12_group_GROUP_ROOT / f'{stem}_stat-logpFWEclusterSize_p001.nii.gz') if cluster_inference else None}
    mask = u12_group_analysis_mask(subjects, task, name)
    model = u12_SecondLevelModel(mask_img=mask, smoothing_fwhm=None).fit(maps, design_matrix=design)
    parametric = model.compute_contrast('intercept', output_type='all')
    perm = u12_non_parametric_inference(maps, design_matrix=design, second_level_contrast='intercept', mask=mask, smoothing_fwhm=None, n_perm=u12_group_N_PERM, two_sided_test=True, random_state=u12_group_RANDOM_STATE, n_jobs=2, threshold=u12_group_CLUSTER_FORMING_P if cluster_inference else None, tfce=False, verbose=0)
    u12_group_GROUP_ROOT.mkdir(parents=True, exist_ok=True)
    u12_group_save_float(mask, u12_group_GROUP_ROOT / f'{stem}_mask.nii.gz')
    u12_group_save_float(parametric['effect_size'], u12_group_GROUP_ROOT / f'{stem}_stat-effect.nii.gz')
    u12_group_save_float(parametric['z_score'], u12_group_GROUP_ROOT / f'{stem}_stat-z.nii.gz')
    logp_img = perm['logp_max_t'] if isinstance(perm, dict) else perm
    u12_group_save_float(parametric['stat'], u12_group_GROUP_ROOT / f'{stem}_stat-t.nii.gz')
    u12_group_save_float(logp_img, u12_group_GROUP_ROOT / f'{stem}_stat-logpFWEmaxT.nii.gz')
    cluster_logp_img = None
    if cluster_inference:
        cluster_perm = perm
        cluster_logp_img = cluster_perm['logp_max_size']
        u12_group_save_float(cluster_perm['size'], u12_group_GROUP_ROOT / f'{stem}_stat-clusterSize_p001.nii.gz')
        u12_group_save_float(cluster_logp_img, u12_group_GROUP_ROOT / f'{stem}_stat-logpFWEclusterSize_p001.nii.gz')
    meta = {'code_sha256': u12_source_signature(), 'task': task, 'analysis': name, 'first_level_model_version': u12_MODEL_VERSIONS[task], 'nilearn_version': u12_nilearn.__version__, 'subjects_n': len(subjects), 'subjects': subjects, 'input_signatures': input_signatures, 'design_columns': list(design.columns), 'inference_primary': 'two-sided permutation cluster-size FWE; CDT p < .001; cluster FWE p < .05', 'inference_complementary': 'two-sided permutation max-t voxelwise FWE p < .05', 'n_permutations': u12_group_N_PERM, 'random_state': u12_group_RANDOM_STATE, 'alpha_fwe': u12_group_FWE_ALPHA, 'cluster_inference': bool(cluster_inference), 'cluster_forming_p': u12_group_CLUSTER_FORMING_P if cluster_inference else None, 'cluster_connectivity': 6, 'group_mask_subject_coverage': 1.0, 'group_mask_definition': 'intersection of all supplied valid individual analysis masks; connected component'}
    meta_path.write_text(u12_json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
    return {'name': name, 'task': task, 'subjects': subjects, 'z': u12_nib.load(u12_group_GROUP_ROOT / f'{stem}_stat-z.nii.gz'), 'logp': u12_nib.load(u12_group_GROUP_ROOT / f'{stem}_stat-logpFWEmaxT.nii.gz'), 'cluster_logp': u12_nib.load(u12_group_GROUP_ROOT / f'{stem}_stat-logpFWEclusterSize_p001.nii.gz') if cluster_inference else None}


In [ ]:
def u12_group_atlas_lookup_setup():
    atlas = u12_datasets.fetch_atlas_aal(version='SPM12')
    img = u12_nib.load(atlas.maps)
    labels = {int(i): label for (i, label) in zip(atlas.indices, atlas.labels)}
    return (img, labels)


In [ ]:
def u12_group_clusters(result: dict, atlas_img, atlas_labels, inference: str='voxel_maxT') -> u12_pd.DataFrame:
    z_img = result['z']
    lp_img = result['logp'] if inference == 'voxel_maxT' else result['cluster_logp']
    if lp_img is None:
        return u12_pd.DataFrame()
    z = u12_np.asarray(z_img.dataobj)
    lp = u12_np.asarray(lp_img.dataobj)
    sig = u12_np.isfinite(lp) & (lp >= -u12_math.log10(u12_group_FWE_ALPHA))
    atlas_res = u12_resample_to_img(atlas_img, z_img, interpolation='nearest', force_resample=True, copy_header=True)
    atlas_data = u12_np.asarray(atlas_res.dataobj).astype(int)
    voxel_mm3 = float(abs(u12_np.linalg.det(z_img.affine[:3, :3])))
    rows = []
    for (sign, sign_mask) in (('positive', z > 0), ('negative', z < 0)):
        (lab, nlab) = u12_ndimage.label(sig & sign_mask, structure=u12_ndimage.generate_binary_structure(3, 1))
        for cid in range(1, nlab + 1):
            vox = u12_np.argwhere(lab == cid)
            vals = z[tuple(vox.T)]
            peak_local = int(u12_np.argmax(vals) if sign == 'positive' else u12_np.argmin(vals))
            ijk = vox[peak_local]
            xyz = u12_nib.affines.apply_affine(z_img.affine, ijk)
            peak_lp = float(lp[tuple(ijk)])
            atlas_index = int(atlas_data[tuple(ijk)])
            rows.append({'task': result['task'], 'analysis': result['name'], 'N': len(result['subjects']), 'inference': inference, 'direction': sign, 'cluster_id': cid, 'cluster_voxels': len(vox), 'cluster_mm3': int(round(len(vox) * voxel_mm3)), 'peak_region_AAL': atlas_labels.get(atlas_index, 'Unlabeled'), 'peak_x': round(float(xyz[0]), 1), 'peak_y': round(float(xyz[1]), 1), 'peak_z': round(float(xyz[2]), 1), 'peak_Z': round(float(z[tuple(ijk)]), 3), 'peak_p_FWE': round(10 ** (-peak_lp), 6)})
    return u12_pd.DataFrame(rows, columns=['task', 'analysis', 'N', 'inference', 'direction', 'cluster_id', 'cluster_voxels', 'cluster_mm3', 'peak_region_AAL', 'peak_x', 'peak_y', 'peak_z', 'peak_Z', 'peak_p_FWE'])


In [ ]:
import sys as u12_sys
import argparse as u12_argparse
import json as u12_json
import nibabel as u12_nib
import numpy as u12_np
import pandas as u12_pd
from scipy import ndimage as u12_ndimage, stats as u12_stats
from nilearn.glm.first_level import make_first_level_design_matrix as u12_make_first_level_design_matrix


In [ ]:
def u12_alignment_audit_first():
    rows = []
    for sid in u12_subjects():
        for task in u12_CONTRAST:
            old = u12_pd.read_csv(u12_qc_path(sid, task), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            if not old.model_version.eq(u12_glm_MODEL_VERSIONS[task]).all():
                raise ValueError(f'Previous GLM version does not match: {sid} {task}')
            if u12_new_first_complete(sid, task):
                signature = u12_glm_input_signature(sid, task)
                if 'input_signature' not in old or not old.input_signature.eq(signature).all():
                    raise ValueError(f'First-level GLM input signature is outdated; refit required: {sid} {task}')
            for (i, run) in enumerate([None] if task == 'stroop' else ['01', '02']):
                p = u12_glm_paths_for(sid, task, run)
                for key in ('bold', 'mask', 'confounds', 'metadata'):
                    if not p[key].exists():
                        raise FileNotFoundError(p[key])
                ev = u12_glm_reconstruct_events(sid, task, None) if task == 'stroop' else u12_glm_load_events(p['events'], task)
                if not u12_np.allclose(ev.duration, 1.0 if task == 'stroop' else 0.5):
                    raise ValueError('Event durations do not match')
                (cf, info) = u12_glm_load_confounds(p['confounds'])
                (tr, ref) = u12_glm_acquisition(p['metadata'])
                frames = u12_np.arange(len(cf)) * tr + ref * tr
                dm = u12_make_first_level_design_matrix(frames, events=ev, hrf_model='glover + derivative', drift_model='cosine', high_pass=1 / 128, add_regs=cf.to_numpy(), add_reg_names=cf.columns.tolist())
                q = old.iloc[i]
                if list(dm.columns) != str(q['columns']).split('|') or u12_np.linalg.matrix_rank(dm) != int(q.design_rank):
                    raise ValueError(f'Observed design does not match the previous model; refit required: {sid} {task} {run}')
                if len(cf) != int(q.n_timepoints) or u12_nib.load(p['bold']).shape[-1] != len(cf):
                    raise ValueError('Volume count does not match')
                raw = u12_pd.read_csv(p['confounds'], sep=',' if u12_Path(p['confounds']).suffix.lower() == '.csv' else '\t', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
                fd = raw.framewise_displacement.fillna(0).to_numpy()
                dv = raw.std_dvars.fillna(0).to_numpy()
                nss = raw.filter(regex='^non_steady_state_outlier').fillna(0).to_numpy()
                spikes = (fd > 0.5) | (dv > 1.5)
                all_spikes = spikes | (nss.any(axis=1) if nss.shape[1] else False)
                if (fd > 0.5).mean() > 0.2:
                    raise ValueError(f'Task run exceeds the manuscript motion threshold: {sid} {task} {run}')
                if int(spikes.sum()) != int(q.n_spikes):
                    raise ValueError('Previous GLM spike count does not match')
                rows.append(dict(subject_id=sid, task=task, run=run or 'single', dof=len(cf) - int(q.design_rank), mean_fd_mm=float(fd.mean()), fd_gt_05_fraction=float((fd > 0.5).mean()), all_spikes_fraction=float(all_spikes.mean()), n_volumes=len(cf), design_rank=int(q.design_rank), model_version=q.model_version, input_signature=u12_fingerprint([p[k] for k in ('bold', 'mask', 'confounds', 'metadata')] + ([p['events']] if task == 'gonogo' else [u12_glm_raw_log_path(sid, task, None)]))))
            (t, valid, img, dof) = u12_t_map(sid, task)
            if not valid.any() or not u12_np.isfinite(t).all():
                raise ValueError('Invalid subject-level t image')
            for kind in ('effect', 'variance', 'z'):
                im = u12_nib.load(u12_first_path(sid, task, kind))
                if im.get_data_dtype().kind != 'f':
                    raise ValueError('Previous image has integer-quantization problems')
        print('Validate first-level GLM', sid, flush=True)
    df = u12_pd.DataFrame(rows)
    u12_NEW.mkdir(parents=True, exist_ok=True)
    df.to_csv(u12_NEW / 'observed_inputs_and_model_validation.csv', index=False)
    u12_atomic_json(u12_NEW / 'first_level_GLM_validation.json', dict(status='validated_current_round_maps', subjects=u12_subjects(), runs=len(df), hrf='glover + derivative', ar1=True, smoothing_fwhm_mm=6, highpass_hz=1 / 128, duration_s={'stroop': 1, 'gonogo': 0.5}, note='Check current images against observed inputs, design columns and rank, spike counts, file precision, and software versions; reuse only completed results with matching signatures'))
    return df


In [ ]:
def u12_alignment_group():
    global u12_group_effect_path
    u12_group_effect_path = lambda sid, task: u12_first_path(sid, task, 'effect')
    u12_group_GROUP_ROOT.mkdir(parents=True, exist_ok=True)
    for task in u12_CONTRAST:
        stem = f'task-{task}_analysis-primary'
        result = u12_group_infer('primary', u12_subjects(), task, cluster_inference=True)
        for p in u12_group_GROUP_ROOT.glob(stem + '*.nii.gz'):
            im = u12_nib.load(p)
            if not p.name.endswith('_mask.nii.gz') and im.get_data_dtype() != u12_np.dtype('float32'):
                raise ValueError('Group image precision is incorrect')
        print('Group-level inference complete (eligible results reused)', task, flush=True)
    (atlas, labels) = u12_group_atlas_lookup_setup()
    results = [u12_group_infer('primary', u12_subjects(), t, cluster_inference=True) for t in u12_CONTRAST]
    for (key, inference) in [('permutation cluster FWE', 'cluster_size_FWE_p001'), ('maxT voxelwise FWE', 'voxel_maxT')]:
        u12_save_table(u12_pd.concat([u12_group_clusters(r, atlas, labels, inference) for r in results], ignore_index=True), u12_TASK, 'group_level_' + key)


In [ ]:
def u12_alignment_overlap():
    root = u12_NEW
    zs = [u12_nib.load(root / f'task-{t}_analysis-primary_stat-z.nii.gz') for t in u12_CONTRAST]
    sig = []
    for (t, z) in zip(u12_CONTRAST, zs):
        lp = u12_nib.load(root / f'task-{t}_analysis-primary_stat-logpFWEclusterSize_p001.nii.gz')
        if z.shape != lp.shape or not u12_np.allclose(z.affine, lp.affine):
            raise ValueError('Group image geometries do not match')
        sig.append((u12_np.asarray(z.dataobj) > 0) & (u12_np.asarray(lp.dataobj) >= -u12_np.log10(0.05)))
    both = sig[0] & sig[1]
    out = u12_np.where(both, u12_np.minimum(u12_np.asarray(zs[0].dataobj), u12_np.asarray(zs[1].dataobj)), 0)
    u12_nib.save(u12_nib.Nifti1Image(out.astype('float32'), zs[0].affine), root / 'intersection_of_positive_cluster_FWE_maps.nii.gz')
    (labels, n) = u12_ndimage.label(both, u12_np.ones((3, 3, 3)))
    rows = []
    for i in range(1, n + 1):
        ijk = u12_np.argwhere(labels == i)
        peak = ijk[u12_np.argmax(out[tuple(ijk.T)])]
        xyz = u12_nib.affines.apply_affine(zs[0].affine, peak)
        rows.append(dict(cluster=i, voxels=len(ijk), volume_mm3=len(ijk) * abs(u12_np.linalg.det(zs[0].affine[:3, :3])), x=xyz[0], y=xyz[1], z=xyz[2], min_z=out[tuple(peak)]))
    u12_save_table(u12_pd.DataFrame(rows), u12_TASK, 'shared_positive_activation_corrected_map_intersection')
    u12_atomic_json(root / 'intersection_inference_notes.json', dict(voxels=int(both.sum()), components=n, operation='Intersection of separately corrected positive cluster-FWE maps', interpretation='Each voxel in the intersection belongs to a significant cluster in each task; the intersection components were not separately FWE-tested and do not constitute a voxelwise minimum-statistic conjunction'))
    print('Shared positive activation voxels', int(both.sum()), flush=True)


In [ ]:
import matplotlib.pyplot as u12_plt
from matplotlib.text import Text as u12_Text
from matplotlib.ticker import FixedLocator as u12_FixedLocator, FixedFormatter as u12_FixedFormatter


In [ ]:
def u12_apa_plain_apa(text, bold=False):
    task_pattern = 'Go/No-Go|GoNoGo|Stroop|GNG(?: [12])?'

    def fragment(part):

        def task_math(match):
            cmd = 'mathbfit' if bold else 'mathit'
            word = match[0].replace(' ', chr(92) + ' ')
            pieces = word.split('-')
            return '$' + (chr(92) + 'text{-}').join((chr(92) + cmd + '{' + piece + '}' for piece in pieces)) + '$'
        part = u12_re.sub(task_pattern, task_math, part)
        tokens = u12_re.split('(\\$[^$]*\\$)', part)
        for (i, token) in enumerate(tokens):
            if not token.startswith('$'):
                tokens[i] = u12_re.sub('\\b(r|p|t|z|Z|N|n|M|SD|SE|df|d|F|v|a)\\b', lambda m: '$' + m[0] + '$', token)
        return ''.join(tokens)
    pieces = u12_re.split('(\\$[^$]*\\$)', text)
    return ''.join((part if part.startswith('$') else fragment(part) for part in pieces))


In [ ]:
def u12_apa_apply_apa_style(fig):
    if getattr(fig, '_a8_APA_applied', False):
        return
    fig._a8_APA_applied = True
    for axis in [axis for ax in fig.axes for axis in [ax.xaxis, ax.yaxis]]:
        current = [v.get_text() for v in axis.get_ticklabels()]
        changed = [u12_apa_plain_apa(v) for v in current]
        if current != changed:
            axis.set_major_locator(u12_FixedLocator(axis.get_majorticklocs()))
            axis.set_major_formatter(u12_FixedFormatter(changed))
    for text in fig.findobj(match=u12_Text):
        value = text.get_text()
        if not value:
            continue
        out = u12_apa_plain_apa(value, bold=text.get_fontweight() in ['bold', 'heavy', 700, 800, 900])
        if out != value:
            text.set_text(out)


In [ ]:
import re as u12_re
import numpy as u12_np
import pandas as u12_pd
from matplotlib.transforms import Bbox as u12_Bbox


In [ ]:
def u12_style_apply_final_style(fig, root, name):
    if getattr(fig, '_a8_final_style_applied', False):
        return
    fig._a8_final_style_applied = True
    ax = fig.axes[0]
    if name == 'Figure_S1_representative_design_matrix_n106':
        fig._a8_tight_export = True
    elif name == 'Figure_S1_head_motion_distribution_n106':
        for c in ax.collections:
            c.set_alpha(0.9)
            c.set_edgecolors('black')
            c.set_linewidths(0.25)
    elif name.startswith('Figure_S4_'):
        top = 304 if 'Stroop' in name else 219
        fig._a8_export_bbox = u12_Bbox.from_extents(150 / 600, 38 / 600, 4191 / 600, (2100 - top) / 600)
    elif name.startswith('Figure_S2_'):
        if 'individual_Dice_distribution' in name:
            for c in ax.collections:
                c.set_alpha(0.9)
                c.set_edgecolors('black')
                c.set_linewidths(0.25)
        elif 'activation_counts' in name:
            fig.legends[0].set_bbox_to_anchor((0.55, 0.06), transform=fig.transFigure)
    elif name.startswith('classical_brain_behavior_'):
        fig.set_size_inches(2.9, 1.9)
        rest = 'rest_connectivity' in name
        fig.subplots_adjust(left=0.37 if rest else 0.25, right=0.98, bottom=0.3, top=0.98)
        ax.xaxis.label.set_fontsize(6.5)
        ax.tick_params(labelsize=5.5)
        legend = fig.legends[0]
        legend.set_bbox_to_anchor((0.55, 0.01), transform=fig.transFigure)
        for text in legend.get_texts():
            text.set_fontsize(5.5)
        if rest:
            data = u12_pd.read_excel(u12_style_ROOT / '1.resting_state/tables/brain_behavior_4_connectivity_by_2_behavior_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            key = 'composite'
            order = ['shared_core', 'stroop_priority', 'gonogo_priority', 'bridge']
        else:
            data = u12_pd.read_excel(u12_style_ROOT / '2.task_fmri/tables/brain_behavior_5ROI_by_2task_by_2behavior_ALE_3D.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            data = data[data.task.eq('stroop' if 'Stroop' in name else 'gonogo')]
            key = 'roi_id'
            order = u12_ROIS
        for (k, outcome) in enumerate(['stroop_rt_cost_ms', 'nogo_accuracy']):
            for (j, (_, row)) in enumerate(data[data.outcome.eq(outcome)].set_index(key).loc[order].iterrows()):
                xpos = float(row.CI95_high) + 0.025
                if xpos < 0 and xpos + 0.15 > 0:
                    xpos = 0.025
                ax.text(xpos, j + (k - 0.5) * 0.27, f'{float(row.r):.3f}', ha='left', va='center', fontsize=5.2, color=['#3C5488', '#E64B35'][k])
    elif name.startswith('Figure_S7_'):
        fig.set_size_inches(3.6, 2.1)
        fig.subplots_adjust(left=0.2, right=0.985, bottom=0.2, top=0.89)
        ax.set_ylabel('Cross-task pattern\nsimilarity (Fisher z)', fontsize=7, labelpad=3)
        ax.tick_params(labelsize=6)
        for c in ax.collections:
            if len(c.get_offsets()) > 100:
                c.set_alpha(0.9)
                c.set_edgecolors('black')
                c.set_linewidths(0.25)
        data = u12_pd.read_excel(u12_style_ROOT / '2.task_fmri/tables/independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id')
        for (label, key) in zip(ax.texts, u12_ROIS[:3]):
            val = float(data.loc[key, 'p_positive_FDR_3'])
            label.set_text('$p_{\\mathrm{FDR}} = ' + f'{val:.3f}' + '$')
            label.set_y(1.015)
            label.set_fontsize(7.5)
    elif name in ['Figure_independent_ALE_ROI_activation_n106', 'Figure_independent_ALE_ROI_cross_task_paired_difference_n106', 'Figure_shared_ALE_ROI_unsmoothed_pattern_similarity_n106']:
        base = u12_style_ROOT / '2.task_fmri/tables'
        if name == 'Figure_independent_ALE_ROI_activation_n106':
            one = u12_pd.read_excel(base / 'independent_ALE_ROI_ALE_3D_10_activation_tests.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            for (k, task) in enumerate(u12_CONTRAST):
                for (j, (_, row)) in enumerate(one[one.task.eq(task)].set_index('roi_id').loc[u12_ROIS].iterrows()):
                    if row.p_flip_FDR_10 < 0.05:
                        ax.text(float(row.CI95_high) + 0.025, j + (k - 0.5) * 0.22, '*', fontsize=8, color=['#397A9E', '#B76B3F'][k], ha='left', va='center')
            (lo, hi) = ax.get_xlim()
            ax.set_xlim(lo, hi + 0.06)
        else:
            paired = 'paired' in name
            data = u12_pd.read_excel(base / ('independent_ALE_ROI_ALE_3D_cross_task_paired_tests.xlsx' if paired else 'independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id')
            for (j, key) in enumerate(u12_ROIS if paired else u12_ROIS[:3]):
                q = float(data.loc[key, 'p_FDR_5' if paired else 'p_positive_FDR_3'])
                value = '< 0.001' if q < 0.001 else '= ' + (f'{q:.5f}' if 0.05 < q < 0.0505 else f'{q:.3f}')
                ax.text(0.98, j - 0.26, '$p_{\\mathrm{FDR}} ' + value + '$', transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=7.5)
    elif name in ['Figure_independent_ALE_ROI_resting_connectivity_matrix_n106', 'Figure_independent_ALE_ROI_four_prespecified_connectivity_measures_n106']:
        for text in list(fig.texts):
            text.remove()
        fig._a8_tight_export = True
        if 'matrix' in name:
            fig.set_size_inches(2.8, 2.5)
            fig.subplots_adjust(left=0.21, right=0.81, bottom=0.25, top=0.93)
            fig.axes[1].set_position([0.84, 0.25, 0.025, 0.68])
            fig.axes[1].yaxis.label.set_fontsize(6.5)
            fig.axes[1].tick_params(labelsize=5.5)
            edges = u12_pd.read_excel(u12_style_ROOT / '1.resting_state/tables/functional_connectivity_10_edges_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            lookup = {frozenset(row.edge.split('__')): row for (_, row) in edges.iterrows()}
            for text in ax.texts:
                (j, i) = map(int, text.get_position())
                row = lookup[frozenset([u12_ROIS[i], u12_ROIS[j]])]
                text.set_text(f"{float(row['mean']):.2f}" + ('*' if row.p_flip_FDR_10 < 0.05 else ''))
                text.set_fontsize(6.5)
            ax.tick_params(labelsize=6, length=2)
        else:
            fig.set_size_inches(2.9, 1.9)
            fig.subplots_adjust(left=0.38, right=0.98, bottom=0.23, top=0.98)
            ax.xaxis.label.set_fontsize(6.5)
            ax.tick_params(labelsize=6, length=2)


In [ ]:
from functools import lru_cache as u12_lru_cache
import tempfile as u12_tempfile
import hashlib as u12_hashlib
import json as u12_json
import warnings as u12_warnings
import numpy as u12_np
import nibabel as u12_nib
import matplotlib as u12_matplotlib
import matplotlib.pyplot as u12_plt
from matplotlib.colors import Normalize as u12_Normalize, ListedColormap as u12_ListedColormap, BoundaryNorm as u12_BoundaryNorm, LinearSegmentedColormap as u12_LinearSegmentedColormap
from nilearn import datasets as u12_datasets, surface as u12_surface, plotting as u12_plotting
u12_brain_POSITIVE_STOPS = [(0.0, '#B2182B'), (0.45, '#E64B35'), (0.75, '#F39B2F'), (1.0, '#FDE34F')]
u12_brain_BRAIN_POSITIVE_CMAP = u12_LinearSegmentedColormap.from_list('a8_positive', u12_brain_POSITIVE_STOPS, N=4096)
u12_brain_BRAIN_SIGNED_CMAP = u12_LinearSegmentedColormap.from_list('a8_signed', [(0.0, '#55C9DD'), (0.25, '#237BC0'), (0.49999, '#24489E'), (0.5, '#F7F7F7')] + [(0.50001 + 0.49999 * x, c) for (x, c) in u12_brain_POSITIVE_STOPS], N=8192)
u12_brain_ROI_COLORS = ['#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F']
u12_brain_ROI_LABELS = ['SMA', 'rAI/FO', 'lAI', 'lIPS', 'rIFG/PMC']
u12_brain_VIEWS = [('left', 'lateral', 'L lateral'), ('left', 'medial', 'L medial'), ('right', 'medial', 'R medial'), ('right', 'lateral', 'R lateral')]


In [ ]:
def u12_brain_anatomical_shading(ax, mesh_path, values, sulc, hemi, view, categorical=False):
    (faces, luminance) = u12_brain_cortical_luminance(str(mesh_path), str(sulc), hemi, view)
    fv = values[faces]
    active = u12_np.median(fv, axis=1) > 0.5 if categorical else (u12_np.all(fv > 0, axis=1) | u12_np.all(fv < 0, axis=1)) & (u12_np.abs(fv.mean(axis=1)) >= 1e-08)
    changed = 0
    for coll in ax.collections:
        original = coll._facecolor3d.copy()
        if len(original) != len(faces):
            continue
        result = original.copy()
        result[~active, :3] = luminance[~active, None]
        assert u12_np.array_equal(result[active], original[active])
        assert u12_np.array_equal(result[:, 3], original[:, 3])
        edges = coll._edgecolor3d.copy()
        assert edges.shape == original.shape
        edges[~active, :3] = luminance[~active, None]
        assert u12_np.array_equal(edges[active], coll._edgecolor3d[active])
        coll.set_facecolors(result)
        coll.set_edgecolors(edges)
        changed += 1
    assert changed == 1


In [ ]:
def u12_brain_strict_face_mean(values):
    active = u12_np.all(values > 0) | u12_np.all(values < 0)
    return float(values.mean()) if active else 0.0


In [ ]:
def u12_brain_project(img, mask, categorical=False):
    h = u12_hashlib.sha256()
    for im in (img, mask):
        h.update(u12_np.asanyarray(im.dataobj).tobytes())
        h.update(im.affine.tobytes())
    h.update(u12_source_signature().encode())
    h.update(u12_sha(u12_input_path('MNI152_display_reference.nii.gz')).encode())
    h.update(str(categorical).encode())
    h.update(u12_sha(u12_brain_inverse_transform()).encode())
    key = h.hexdigest()
    dest = u12_brain_CACHE / (key + '.npz')
    if dest.exists():
        with u12_np.load(dest, allow_pickle=False) as z:
            return {k: z[k].copy() for k in z.files}
    import ants as ants
    fs = u12_brain_meshes()
    with u12_tempfile.TemporaryDirectory(prefix='a8_surface_') as tmp:
        tmp = u12_Path(tmp)
        u12_nib.save(img, tmp / 'values.nii.gz')
        u12_nib.save(mask, tmp / 'mask.nii.gz')
        fixed = ants.image_read(str(u12_input_path('MNI152_display_reference.nii.gz')))
        transform = [str(u12_brain_inverse_transform())]
        warped = ants.apply_transforms(fixed=fixed, moving=ants.image_read(str(tmp / 'values.nii.gz')), transformlist=transform, interpolator='genericLabel' if categorical else 'linear')
        valid = ants.apply_transforms(fixed=fixed, moving=ants.image_read(str(tmp / 'mask.nii.gz')), transformlist=transform, interpolator='genericLabel')
        ants.image_write(warped, str(tmp / 'values6.nii.gz'))
        ants.image_write(valid, str(tmp / 'mask6.nii.gz'))
        result = {}
        for hemi in ('left', 'right'):
            with u12_warnings.catch_warnings():
                u12_warnings.filterwarnings('ignore', message='Mean of empty slice')
                values = u12_surface.vol_to_surf(str(tmp / 'values6.nii.gz'), fs['pial_' + hemi], inner_mesh=fs['white_' + hemi], kind='depth', depth=[0.5], interpolation='nearest_most_frequent' if categorical else 'linear', mask_img=None if categorical else str(tmp / 'mask6.nii.gz'))
            result[hemi] = u12_np.nan_to_num(values, nan=0.0)
            if categorical:
                assert set(u12_np.unique(result[hemi])).issubset(set(range(6)))
    temp = dest.with_suffix('.tmp.npz')
    u12_np.savez_compressed(temp, **result)
    temp.replace(dest)
    (u12_brain_CACHE / (key + '.json')).write_text(u12_json.dumps(dict(purpose='visualization only', template_transform=str(u12_brain_inverse_transform().name), transform_sha256=u12_sha(u12_brain_inverse_transform()), native_space='MNI152NLin2009cAsym', surface_frame='MNI152NLin6Asym', mesh='fsaverage6; mid-ribbon samples', depth=0.5, categorical=categorical, nonzero_vertices={k: int(u12_np.count_nonzero(v)) for (k, v) in result.items()}), indent=2))
    return result


In [ ]:
def u12_brain_surface_row(fig, rect, textures, cmap, vmin, vmax, categorical=False, labels=True):
    fs = u12_brain_meshes()
    (x, y, w, h) = rect
    gap = 0.012 * w
    aw = (w - 3 * gap) / 4
    axes = []
    for (i, (hemi, view, name)) in enumerate(u12_brain_VIEWS):
        ax = fig.add_axes([x + i * (aw + gap), y, aw, h], projection='3d')
        opts = dict(surf_mesh=fs['pial_' + hemi], bg_map=fs['sulc_' + hemi], hemi=hemi, view=view, engine='matplotlib', cmap=cmap, colorbar=False, vmin=vmin, vmax=vmax, bg_on_data=not categorical and vmin == 0 and (vmax == 1), alpha=1, axes=ax, figure=fig)
        if categorical:
            u12_plotting.plot_surf_roi(roi_map=textures[hemi], avg_method='median', threshold=0.5, **opts)
        else:
            u12_plotting.plot_surf_stat_map(stat_map=textures[hemi], threshold=1e-08, symmetric_cbar=vmin < 0, avg_method=u12_brain_strict_face_mean, **opts)
        u12_brain_anatomical_shading(ax, fs['pial_' + hemi], textures[hemi], fs['sulc_' + hemi], hemi, view, categorical)
        ax.set_box_aspect(None, zoom=1.5)
        for collection in ax.collections:
            collection.set_rasterized(True)
        if labels:
            fig.text(x + (i + 0.5) * (aw + gap) - gap / 2, y + 0.012, name, ha='center', va='center', fontsize=7)
        axes.append(ax)
    return axes


In [ ]:
def u12_brain_axial_row(fig, rect, img, cuts, cmap, norm, categorical=False):
    im = u12_nib.as_closest_canonical(img)
    bg = u12_brain_background()
    assert im.shape == bg.shape and u12_np.allclose(im.affine, bg.affine), 'Axial image/background grids differ'
    values = u12_np.asarray(im.dataobj)
    anat = u12_np.where(u12_brain_background_mask(), u12_np.asarray(bg.dataobj), 0.0)
    shape = im.shape
    xyz0 = u12_nib.affines.apply_affine(im.affine, [-0.5, -0.5, 0])
    xyz1 = u12_nib.affines.apply_affine(im.affine, [shape[0] - 0.5, shape[1] - 0.5, 0])
    extent = [xyz0[0], xyz1[0], xyz0[1], xyz1[1]]
    vmax_bg = float(u12_np.quantile(anat[anat > 0], 0.995))
    (x, y, w, h) = rect
    gap = 0.018 * w
    aw = (w - gap * (len(cuts) - 1)) / len(cuts)
    axes = []
    for (i, z) in enumerate(cuts):
        ax = fig.add_axes([x + i * (aw + gap), y, aw, h])
        k = int(round(u12_nib.affines.apply_affine(u12_np.linalg.inv(im.affine), [0, 0, z])[2]))
        a = anat[:, :, k].T
        v = values[:, :, k].T
        ax.imshow(u12_np.ma.masked_less_equal(a, 0), origin='lower', extent=extent, cmap='gray', vmin=0, vmax=vmax_bg, interpolation='bilinear')
        shown = u12_np.ma.masked_where(abs(v) < 1e-09, v)
        ax.imshow(shown, origin='lower', extent=extent, cmap=cmap, norm=norm, interpolation='nearest')
        ax.set(xlim=(-80, 80), ylim=(-112, 80))
        ax.set_axis_off()
        fig.text(x + i * (aw + gap) + aw / 2, y - 0.015, f'z = {im.affine[2, 3] + k * im.affine[2, 2]:g} mm', fontsize=7, ha='center', va='center')
        if i == 0:
            ax.text(0.03, 0.94, 'L', transform=ax.transAxes, fontsize=7, va='top')
            ax.text(0.91, 0.94, 'R', transform=ax.transAxes, fontsize=7, va='top')
        axes.append(ax)
    return axes


In [ ]:
def u12_brain_set_alignment(fig, rows):
    axes = []
    ids = []
    groups = []
    for (i, row) in enumerate(rows):
        names = [f'row{i}_view{j}' for j in range(len(row))]
        axes += row
        ids += names
        groups.append(names)
    fig._a8_alignment_options = dict(axes=axes, panel_ids=ids, row_groups=groups, column_groups=[])


In [ ]:
def u12_brain_rows(fig, upper, lower, img, cmap=u12_brain_BRAIN_SIGNED_CMAP, vmin=-6, vmax=6, cuts=(-20, 6, 20, 40, 60), categorical=False, annotations=None):
    data = u12_np.asarray(img.dataobj)
    mask = u12_nib.Nifti1Image((abs(data) > 1e-09).astype('uint8'), img.affine)
    tex = u12_brain_project(img, mask, categorical=categorical)
    top = u12_brain_surface_row(fig, upper, tex, cmap, vmin, vmax, categorical=categorical)
    norm = u12_BoundaryNorm(u12_np.arange(0.5, 6, 1), 5) if categorical else u12_Normalize(vmin, vmax)
    bottom = u12_brain_axial_row(fig, lower, img, cuts, cmap, norm, categorical=categorical)
    if annotations is not None:
        u12_annotation_annotate_rows(fig, top, bottom, img, cuts, tex, categorical, annotations)
    return [top, bottom]


In [ ]:
def u12_brain_horizontal_bar(fig, rect, cmap, vmin, vmax, label='Z statistic'):
    cax = fig.add_axes(rect)
    cb = fig.colorbar(u12_plt.cm.ScalarMappable(norm=u12_Normalize(vmin, vmax), cmap=cmap), cax=cax, orientation='horizontal', extend='both' if vmin < 0 else 'max')
    cb.set_ticks(u12_np.linspace(vmin, vmax, 5 if vmin < 0 else 4))
    from matplotlib.ticker import FormatStrFormatter as FormatStrFormatter
    cb.ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    cb.ax.tick_params(labelsize=6, length=2, width=0.5, pad=1)
    fig.text(rect[0] + rect[2] + 0.025, rect[1] + rect[3] / 2, label, fontsize=7, va='center')
    return cb


In [ ]:
def u12_brain_stat_figure(img, title, subtitle, cmap=u12_brain_BRAIN_SIGNED_CMAP, vmin=-6, vmax=6, label_scope='positive'):
    fig = u12_plt.figure(figsize=(7.2, 3.5))
    panel_rows = u12_brain_rows(fig, (0.01, 0.5, 0.98, 0.4), (0.01, 0.13, 0.98, 0.29), img, cmap, vmin, vmax, annotations=label_scope)
    fig.text(0.5, 0.965, title, fontsize=10, fontweight='bold', va='top', ha='center')
    u12_brain_horizontal_bar(fig, (0.3, 0.055, 0.38, 0.02), cmap, vmin, vmax, label='Minimum Z' if vmin == 0 else 'Z statistic')
    fig._a8_caption = subtitle
    u12_brain_set_alignment(fig, panel_rows)
    return fig


In [ ]:
from functools import lru_cache as u12_lru_cache
import numpy as u12_np
import nibabel as u12_nib
from mpl_toolkits.mplot3d import proj3d as u12_proj3d
u12_annotation_LABELS = {1: 'SMA', 2: 'rAI/FO', 3: 'lAI', 4: 'lIPS', 5: 'rIFG/PMC'}
u12_annotation_FULL_NAMES = {1: 'supplementary motor area', 2: 'right anterior insula / frontal operculum', 3: 'left anterior insula', 4: 'left intraparietal sulcus', 5: 'right inferior frontal gyrus / premotor cortex'}


In [ ]:
@u12_lru_cache(maxsize=1)
def u12_annotation_roi_display():
    (masks, ref) = u12_roi_masks()
    labels = u12_np.zeros(ref.shape, dtype='uint8')
    for (i, r) in enumerate(u12_ROIS, 1):
        labels[masks[r]] = i
    img = u12_nib.Nifti1Image(labels, ref.affine)
    mask = u12_nib.Nifti1Image((labels > 0).astype('uint8'), ref.affine)
    return (masks, ref, u12_brain_project(img, mask, categorical=True))


In [ ]:
def u12_annotation_not_occluded(index, projected, faces):
    triangles = projected[faces]
    p = triangles[index].mean(0)
    box = (triangles[:, :, :2].min(1) <= p[:2]).all(1) & (triangles[:, :, :2].max(1) >= p[:2]).all(1)
    ids = u12_np.flatnonzero(box)
    ids = ids[ids != index]
    if not len(ids):
        return True
    tri = triangles[ids]
    a = tri[:, 0, :2]
    v = tri[:, 1, :2] - a
    u = tri[:, 2, :2] - a
    r = p[:2] - a
    det = v[:, 0] * u[:, 1] - v[:, 1] * u[:, 0]
    good = u12_np.abs(det) > 1e-12
    s = u12_np.divide(r[:, 0] * u[:, 1] - r[:, 1] * u[:, 0], det, out=u12_np.zeros(len(ids)), where=good)
    t = u12_np.divide(v[:, 0] * r[:, 1] - v[:, 1] * r[:, 0], det, out=u12_np.zeros(len(ids)), where=good)
    inside = good & (s >= -1e-07) & (t >= -1e-07) & (s + t <= 1 + 1e-07)
    depth = tri[:, 0, 2] + s * (tri[:, 1, 2] - tri[:, 0, 2]) + t * (tri[:, 2, 2] - tri[:, 0, 2])
    return not u12_np.any(inside & (depth < p[2] - 1e-05))


In [ ]:
def u12_annotation_arrow(ax, text, xy, pos, ha, xycoords='data'):
    return ax.annotate(text, xy=xy, xycoords=xycoords, xytext=pos, textcoords='axes fraction', fontsize=6.5, color='#161616', ha=ha, va='top', annotation_clip=False, arrowprops=dict(arrowstyle='->', lw=0.8, color='#161616', mutation_scale=6, shrinkA=2.5, shrinkB=0))


In [ ]:
def u12_annotation_surface_label(ax, values, roi_texture, idx, hemi, view):
    (xyz, faces) = u12_surface.load_surf_mesh(u12_brain_meshes()['pial_' + hemi])
    xyz = u12_np.asarray(xyz, dtype=float)
    fv = values[faces]
    member = (roi_texture[faces] == idx).all(1)
    candidates = u12_np.flatnonzero(member & (fv > 0).all(1))
    if not len(candidates):
        return None
    points = xyz[faces]
    normal = u12_np.cross(points[:, 1] - points[:, 0], points[:, 2] - points[:, 0])
    if u12_np.median(u12_np.sum(normal * (points.mean(1) - xyz.mean(0)), axis=1)) < 0:
        normal = -normal
    (az, el) = u12_np.deg2rad([ax.azim, ax.elev])
    camera = u12_np.array([u12_np.cos(el) * u12_np.cos(az), u12_np.cos(el) * u12_np.sin(az), u12_np.sin(el)])
    candidates = candidates[normal[candidates] @ camera > 0]
    centred = xyz - xyz.mean(0)
    projected = u12_np.column_stack(u12_proj3d.proj_transform(*centred.T, ax.get_proj()))
    order = candidates[u12_np.argsort(-fv[candidates].mean(1), kind='stable')]
    chosen = None
    for face in order[:1000]:
        if u12_annotation_not_occluded(int(face), projected, faces):
            chosen = int(face)
            break
    if chosen is None:
        return None
    anchor = projected[faces[chosen], :2].mean(0)
    pos = (0.94, 0.97) if hemi == 'right' else (0.06, 0.97)
    u12_annotation_arrow(ax, u12_annotation_LABELS[idx], anchor, pos, 'right' if hemi == 'right' else 'left')
    return dict(roi_id=u12_ROIS[idx - 1], label=u12_annotation_LABELS[idx], full_name=u12_annotation_FULL_NAMES[idx], panel='surface', hemisphere=hemi, view=view, face_index=chosen, vertices=faces[chosen].tolist(), displayed_value=float(fv[chosen].mean()), label_membership='All three vertices inside independently projected ALE ROI', visibility='Camera-facing; projected triangle centroid passes occlusion test', anchor_definition='Visible positive face within ROI, not whole-brain cluster peak')


In [ ]:
def u12_annotation_axial_label(axes, img, cuts, masks, idx, categorical=False):
    data = u12_np.asarray(img.dataobj)
    mask = masks[u12_ROIS[idx - 1]]
    valid = mask & (data == idx if categorical else data > 0)
    choices = []
    for (i, z) in enumerate(cuts):
        k = int(round(u12_nib.affines.apply_affine(u12_np.linalg.inv(img.affine), [0, 0, z])[2]))
        plane = valid[:, :, k]
        if plane.any():
            choices.append((i, k, int(plane.sum())))
    if not choices:
        return None
    preferred = [c for c in choices if c[0] == 1] if idx in (2, 3) else []
    (i, k, _) = max(preferred or choices, key=lambda c: c[2])
    ij = u12_np.argwhere(valid[:, :, k])
    v = data[ij[:, 0], ij[:, 1], k]
    if categorical:
        from scipy.ndimage import distance_transform_edt as distance_transform_edt
        dist = distance_transform_edt(valid[:, :, k])
        scores = dist[ij[:, 0], ij[:, 1]]
    else:
        scores = v
    point = ij[int(u12_np.argmax(scores))]
    ijk = (int(point[0]), int(point[1]), int(k))
    assert mask[ijk] and (data[ijk] == idx if categorical else data[ijk] > 0)
    xyz = u12_nib.affines.apply_affine(img.affine, ijk)
    if idx == 1:
        pos = (0.5, 1.14)
        ha = 'center'
    elif xyz[0] < 0:
        pos = (0.02, 1.14)
        ha = 'left'
    else:
        pos = (0.98, 1.14)
        ha = 'right'
    u12_annotation_arrow(axes[i], u12_annotation_LABELS[idx], xyz[:2], pos, ha)
    return dict(roi_id=u12_ROIS[idx - 1], label=u12_annotation_LABELS[idx], full_name=u12_annotation_FULL_NAMES[idx], panel='axial', slice_index=i, voxel_ijk=list(ijk), MNI2009c_xyz_mm=xyz.tolist(), displayed_value=float(data[ijk]), label_membership='Original independent ALE mask contains voxel', anchor_definition='Displayed positive voxel inside ROI on an existing slice; not necessarily cluster peak')


In [ ]:
def u12_annotation_annotate_rows(fig, top, bottom, img, cuts, textures, categorical=False, scope='positive'):
    (masks, ref, roi_texture) = u12_annotation_roi_display()
    assert img.shape == ref.shape and u12_np.allclose(img.affine, ref.affine)
    allowed = (1, 2, 3) if scope == 'shared' else (1, 2, 3, 4, 5)
    records = []
    done = set()
    for (idx, view_index, hemi, view) in [(4, 0, 'left', 'lateral'), (1, 1, 'left', 'medial'), (5, 3, 'right', 'lateral')]:
        if idx not in allowed:
            continue
        record = u12_annotation_surface_label(top[view_index], textures[hemi], roi_texture[hemi], idx, hemi, view)
        if record is not None:
            records.append(record)
            done.add(idx)
    for idx in allowed:
        if idx in done:
            continue
        record = u12_annotation_axial_label(bottom, img, cuts, masks, idx, categorical)
        if record is not None:
            records.append(record)
            done.add(idx)
    if categorical:
        assert done == set(allowed), f'Independent ROI labels missing: {set(allowed) - done}'
    for r in records:
        r['map_role'] = 'independent_ROI_definition' if categorical else 'displayed_positive_map_within_independent_ROI'
    fig._a8_annotations = getattr(fig, '_a8_annotations', []) + records
    return records


In [ ]:
import os as u12_os
import subprocess as u12_subprocess
import matplotlib as u12_matplotlib
u12_matplotlib.use('Agg')
import matplotlib.pyplot as u12_plt
from matplotlib.lines import Line2D as u12_Line2D
import numpy as u12_np
import pandas as u12_pd
u12_plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.7, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'xtick.labelsize': 6, 'ytick.labelsize': 7, 'legend.frameon': False})
u12_PLOT_LABELS = ['SMA', 'rAI/FO', 'lAI', 'lIPS', 'rIFG/PMC']
u12_PLOT_COLORS = {'stroop': '#397A9E', 'gonogo': '#B76B3F'}
u12_PLOT_TASK_FIGURE_NAMES = ['Figure_independent_ALE_ROI_activation_n106', 'Figure_independent_ALE_ROI_cross_task_paired_difference_n106', 'Figure_shared_ALE_ROI_unsmoothed_pattern_similarity_n106']
u12_PLOT_QC_FIGURE_NAMES = ['Figure_S1_representative_design_matrix_n106', 'Figure_S1_head_motion_distribution_n106']
u12_PLOT_COVERAGE_FIGURE_NAMES = ['Figure_S1_Stroop_standard_space_brain_coverage_n106', 'Figure_S1_GoNoGo_standard_space_brain_coverage_n106']
u12_PLOT_MAXT_FIGURE_NAMES = ['Figure_S4_Stroop_voxelwise_maxT_FWE_n106', 'Figure_S4_GoNoGo_voxelwise_maxT_FWE_n106']


In [ ]:
def u12_export(fig, root, name):
    u12_style_apply_final_style(fig, root, name)
    u12_apa_apply_apa_style(fig)
    out = root / 'image'
    out.mkdir(exist_ok=True)
    fig.canvas.draw()
    fig.savefig(out / (name + '.png'), dpi=600, facecolor='white', bbox_inches='tight')
    u12_plt.close(fig)


In [ ]:
def u12_plot_task(figure_names=None):
    base = u12_TASK / 'tables'
    one = u12_pd.read_excel(base / 'independent_ALE_ROI_ALE_3D_10_activation_tests.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    paired = u12_pd.read_excel(base / 'independent_ALE_ROI_ALE_3D_cross_task_paired_tests.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id').loc[u12_ROIS]
    pat = u12_pd.read_excel(base / 'independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id').loc[u12_ROIS[:3]]
    for (index, name) in enumerate(u12_PLOT_TASK_FIGURE_NAMES):
        if figure_names is not None and name not in figure_names:
            continue
        (fig, ax) = u12_plt.subplots(figsize=[(3.3, 2.6), (3.3, 2.3), (3.3, 2.0)][index])
        fig._a8_tight_export = True
        if index == 2:
            fig.subplots_adjust(left=0.2, right=0.98, bottom=0.25, top=0.98)
        else:
            fig.subplots_adjust(left=0.24, right=0.98, bottom=0.25 if index == 0 else 0.2, top=0.98)
        if index == 0:
            for (k, t) in enumerate(u12_CONTRAST):
                d = one[one.task == t].set_index('roi_id').loc[u12_ROIS]
                y = u12_np.arange(5) + (k - 0.5) * 0.22
                ax.errorbar(d['mean'], y, xerr=u12_np.array([d['mean'] - d.CI95_low, d.CI95_high - d['mean']]), fmt='o', color=u12_PLOT_COLORS[t], ms=3.2, lw=0.9, capsize=2)
            ax.set(yticks=u12_np.arange(5), yticklabels=u12_PLOT_LABELS, ylim=(4.6, -0.6), xlabel='Mean voxel t')
            fig.legend([u12_Line2D([0], [0], marker='o', color=u12_PLOT_COLORS[t], lw=0.8, ms=3) for t in u12_CONTRAST], ['Stroop', 'Go/No-Go'], loc='lower center', bbox_to_anchor=(0.6, 0.01), ncol=2, fontsize=7)
        elif index == 1:
            ax.errorbar(paired.mean_difference, u12_np.arange(5), xerr=u12_np.array([paired.mean_difference - paired.CI95_low, paired.CI95_high - paired.mean_difference]), fmt='o', color='#53636B', ms=3.2, lw=0.9, capsize=2)
            ax.set(yticks=u12_np.arange(5), yticklabels=u12_PLOT_LABELS, ylim=(4.6, -0.6), xlabel='Go/No-Go minus Stroop')
        else:
            ax.errorbar(pat['mean'], u12_np.arange(3), xerr=u12_np.array([pat['mean'] - pat.CI95_low, pat.CI95_high - pat['mean']]), fmt='o', color='#725A8E', ms=3.2, lw=0.9, capsize=2)
            ax.set(yticks=u12_np.arange(3), yticklabels=u12_PLOT_LABELS[:3], ylim=(2.6, -0.6), xlabel='Pattern Fisher z')
        ax.axvline(0, color='.7', lw=0.6, ls='--', zorder=0)
        u12_export(fig, u12_TASK, name)


In [ ]:
def u12_brains():
    from nilearn import image as image
    for (task, title) in [('stroop', 'Stroop'), ('gonogo', 'Go/No-Go')]:
        result = u12_group_infer('primary', u12_subjects(), task, cluster_inference=True)
        z = u12_np.asarray(result['z'].dataobj)
        lp = u12_np.asarray(result['cluster_logp'].dataobj)
        sig = lp >= -u12_np.log10(0.05)
        shown = image.new_img_like(result['z'], u12_np.where(sig, z, 0.0))
        vmax = max(3.0, float(u12_np.max(abs(z[sig]))) if sig.any() else 3.0)
        fig = u12_brain_stat_figure(shown, 'Stroop: incongruent − congruent' if task == 'stroop' else 'Go/No-Go: correct No-Go − correct Go', f'Two-sided cluster FWE p < .05; CDT p < .001; N = {len(u12_subjects())}', vmin=-vmax, vmax=vmax)
        name = 'Stroop' if task == 'stroop' else 'GoNoGo'
        u12_export(fig, u12_TASK, f'Figure_task_fmri_{name}main_effect_permutation_cluster_FWE_n106')
    im = u12_nib.load(u12_NEW / 'intersection_of_positive_cluster_FWE_maps.nii.gz')
    vmax = max(3.0, float(u12_np.max(u12_np.asarray(im.dataobj))))
    fig = u12_brain_stat_figure(im, 'Shared positive activation', f'Intersection of separately corrected positive maps; descriptive; N = {len(u12_subjects())}', cmap=u12_brain_BRAIN_POSITIVE_CMAP, vmin=0, vmax=vmax, label_scope='shared')
    u12_export(fig, u12_TASK, 'Figure_shared_positive_activation_descriptive_intersection_n106')


In [ ]:
def u12_qc_summary(design_only=False):
    import nibabel as nib
    from nilearn import plotting as plotting
    from nilearn.glm.first_level import make_first_level_design_matrix as make_first_level_design_matrix
    q = u12_pd.read_csv(u12_NEW / 'observed_inputs_and_model_validation.csv', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    (fig, a) = u12_plt.subplots(figsize=(3.8, 3.4))
    fig.subplots_adjust(left=0.16, right=0.96, bottom=0.33, top=0.95)
    p = u12_glm_paths_for('sub-9001', 'stroop', None)
    (cf, _) = u12_glm_load_confounds(p['confounds'])
    (tr, ref) = u12_glm_acquisition(p['metadata'])
    dm = make_first_level_design_matrix(u12_np.arange(len(cf)) * tr + ref * tr, events=u12_glm_reconstruct_events('sub-9001', 'stroop', None), hrf_model='glover + derivative', drift_model='cosine', high_pass=1 / 128, add_regs=cf.to_numpy(), add_reg_names=cf.columns.tolist())
    x = dm.to_numpy()
    lo = x.min(0)
    scale = x.max(0) - lo
    a.imshow(u12_np.divide(x - lo, scale, out=u12_np.zeros_like(x), where=scale > 0), aspect='auto', cmap='viridis', interpolation='nearest')
    groups = [('Cong.', 'congruent'), ('Incong.', 'incongruent'), ('Neutral', 'neutral'), ('Motion', 'trans_'), ('Drift', 'drift_'), ('Const.', 'constant')]
    ticks = []
    names = []
    for (name, prefix) in groups:
        found = [i for (i, col) in enumerate(dm.columns) if col.startswith(prefix)]
        if found:
            ticks.append(float(u12_np.mean(found)))
            names.append(name)
    a.set(xticks=ticks, xticklabels=names, ylabel='Volume')
    a.tick_params(axis='x', labelsize=6)
    u12_plt.setp(a.get_xticklabels(), rotation=90, rotation_mode='anchor', ha='right', va='center')
    u12_export(fig, u12_TASK, u12_PLOT_QC_FIGURE_NAMES[0])
    if design_only:
        return
    u12_qc_motion()
    u12_qc_coverage()


In [ ]:
def u12_qc_motion():
    q = u12_pd.read_csv(u12_NEW / 'observed_inputs_and_model_validation.csv', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    (fig, b) = u12_plt.subplots(figsize=(2.6, 2.0))
    fig.subplots_adjust(left=0.23, right=0.98, bottom=0.18, top=0.98)
    for (i, (task, run, name)) in enumerate([('stroop', 'single', 'Stroop'), ('gonogo', '01', 'GNG 1'), ('gonogo', '02', 'GNG 2')]):
        sub = q[q.task.eq(task)]
        if task == 'gonogo':
            sub = sub[u12_pd.to_numeric(sub.run).eq(int(run))]
        y = sub.mean_fd_mm.to_numpy()
        offset = u12_np.linspace(-0.17, 0.17, len(y))
        b.scatter(i + offset, y, s=3, color=u12_PLOT_COLORS[task], alpha=0.6, rasterized=True)
        b.plot([i - 0.19, i + 0.19], [u12_np.median(y)] * 2, color='black', lw=1)
    b.set(xticks=range(3), xticklabels=['Stroop', 'GNG 1', 'GNG 2'], ylabel='Mean FD (mm)', xlim=(-0.5, 2.5))
    fig._a8_tight_export = True
    u12_export(fig, u12_TASK, u12_PLOT_QC_FIGURE_NAMES[1])


In [ ]:
def u12_qc_coverage():
    for (task, title, name) in zip(['stroop', 'gonogo'], ['Stroop task', 'Go/No-Go task'], u12_PLOT_COVERAGE_FIGURE_NAMES):
        path = u12_NEW / f'{task}_run_brain_coverage_fraction.nii.gz'
        if not path.exists():
            raise FileNotFoundError(f'Run coverage QC before plotting: {path}')
        im = u12_nib.load(path)
        fig = u12_plt.figure(figsize=(6.4, 2.9))
        fig._a8_tight_export = True
        panels = u12_brain_rows(fig, (0.015, 0.53, 0.97, 0.39), (0.015, 0.145, 0.97, 0.3), im, cmap='viridis', vmin=0, vmax=1, cuts=(-20, 20, 60))
        fig.text(0.5, 0.94, title, fontsize=10, fontweight='bold', ha='center', va='top')
        u12_brain_horizontal_bar(fig, (0.3, 0.055, 0.35, 0.015), 'viridis', 0, 1, 'Participant fraction')
        u12_brain_set_alignment(fig, panels)
        u12_export(fig, u12_TASK, name)


In [ ]:
def u12_max_t():
    from nilearn import image as image
    results = [{'z': u12_nib.load(u12_NEW / f'task-{task}_analysis-primary_stat-z.nii.gz'), 'logp': u12_nib.load(u12_NEW / f'task-{task}_analysis-primary_stat-logpFWEmaxT.nii.gz')} for task in ['stroop', 'gonogo']]
    vmax = max(3.0, max((float(u12_np.max(abs(u12_np.asarray(r['z'].dataobj)))) for r in results)))
    for (name, result) in zip(u12_PLOT_MAXT_FIGURE_NAMES, results):
        z = u12_np.asarray(result['z'].dataobj)
        lp = u12_np.asarray(result['logp'].dataobj)
        sig = lp >= -u12_np.log10(0.05)
        shown = image.new_img_like(result['z'], u12_np.where(sig, z, 0.0))
        fig = u12_plt.figure(figsize=(7.2, 3.5))
        panel_rows = u12_brain_rows(fig, (0.01, 0.5, 0.98, 0.4), (0.01, 0.13, 0.98, 0.29), shown, cmap=u12_brain_BRAIN_SIGNED_CMAP, vmin=-vmax, vmax=vmax, annotations='positive')
        u12_brain_horizontal_bar(fig, (0.3, 0.055, 0.38, 0.02), u12_brain_BRAIN_SIGNED_CMAP, -vmax, vmax)
        u12_brain_set_alignment(fig, panel_rows)
        u12_export(fig, u12_TASK, name)


In [ ]:
from pathlib import Path as u13_Path
import json as u13_json
import nibabel as u13_nib
import numpy as u13_np
import pandas as u13_pd
import json as u13_json
import subprocess as u13_subprocess
import tempfile as u13_tempfile
import nibabel as u13_nib
import numpy as u13_np
import pandas as u13_pd


In [ ]:
def u13_roi_prepare_roi(source, reference, destination, transforms=()):
    (source, reference, destination) = map(u13_Path, (source, reference, destination))
    im = u13_nib.load(source)
    a = u13_np.asarray(im.dataobj)
    if not u13_np.isin(a, [0, 1]).all() or not a.any():
        raise ValueError('Source ROI must be nonempty and binary')
    ref = u13_nib.load(reference)
    transform_paths = [u13_Path(p) for p in transforms]
    if any((not p.is_file() for p in transform_paths)):
        raise FileNotFoundError('An ANTs transform file is missing')
    destination.parent.mkdir(parents=True, exist_ok=True)
    with u13_tempfile.TemporaryDirectory(prefix='a8_roi_') as temp:
        if transform_paths:
            output = u13_Path(temp) / 'mapped.nii.gz'
            command = ['antsApplyTransforms', '-d', '3', '-i', str(source), '-r', str(reference), '-o', str(output), '-n', 'GenericLabel']
            for transform in transform_paths:
                command.extend(['-t', str(transform)])
            u13_subprocess.run(command, check=True, capture_output=True, text=True)
            im = u13_nib.load(output)
        a = u13_np.asarray(im.dataobj)
        if im.shape != ref.shape or not u13_np.allclose(im.affine, ref.affine):
            raise ValueError('ROI must match the analysis grid; supply the required ANTs transforms')
        if not u13_np.isin(a, [0, 1]).all() or not a.any():
            raise ValueError('Mapped ROI must be nonempty and binary')
        u13_nib.save(im, destination)
    return u13_nib.load(destination)


In [ ]:
def u13_roi_define():
    table = u15_table
    file_ref = u15_file_ref
    input_path = u15_input_path
    src = table('ROI_masks.csv', ['roi_id', 'path', 'source_doi', 'independent_of_sample'], ['roi_id'])
    if set(src.roi_id) != set(u13_ROIS):
        raise ValueError('Need the five documented independent ROI masks')
    if not src.independent_of_sample.astype(str).str.lower().isin(['true', '1']).all():
        raise ValueError('ROI independence must be explicitly documented')
    reference = u13_first_path(u13_subjects()[0], 'stroop', 'effect')
    u13_NEW.mkdir(parents=True, exist_ok=True)
    manifest = {}
    rows = []
    cfg = {'source_doi': src.source_doi.tolist(), 'input_manifest_sha256': u13_sha(input_path('ROI_masks.csv')), 'target_space': 'MNI152NLin2009cAsym'}
    u13_atomic_json(u13_ROI_CONFIG, cfg)
    for row in src.itertuples():
        source = file_ref(row.path)
        raw_transforms = getattr(row, 'transforms', None)
        transforms = [] if u13_pd.isna(raw_transforms) or not str(raw_transforms).strip() else u13_json.loads(raw_transforms)
        if not isinstance(transforms, list) or not all((isinstance(p, str) for p in transforms)):
            raise ValueError('transforms must be a JSON array of paths in ANTs CLI order')
        transforms = [file_ref(p) for p in transforms]
        dest = u13_NEW / f'{row.roi_id}_ALE_3D.nii.gz'
        im = u13_roi_prepare_roi(source, reference, dest, transforms)
        a = u13_np.asarray(im.dataobj)
        xyz = u13_nib.affines.apply_affine(im.affine, u13_np.argwhere(a > 0)).mean(axis=0)
        rows.append(dict(roi_id=row.roi_id, target_voxels=int(a.sum()), volume_mm3=float(a.sum() * abs(u13_np.linalg.det(im.affine[:3, :3]))), centroid_x=xyz[0], centroid_y=xyz[1], centroid_z=xyz[2], source=str(source), source_sha256=u13_sha(source), reference_doi=row.source_doi))
        manifest[row.roi_id] = {'sha256': u13_sha(dest), 'source_sha256': u13_sha(source), 'reference_sha256': u13_sha(reference), 'transforms_cli_order': [{'path': str(p), 'sha256': u13_sha(p)} for p in transforms], 'interpolation': 'GenericLabel' if transforms else 'already_on_target_grid'}
    u13_atomic_json(u13_NEW / 'definition_and_provenance.json', {'config_sha256': u13_sha(u13_ROI_CONFIG), 'masks': manifest, 'selection_independent_of_current_sample': True})
    u13_save_table(u13_pd.DataFrame(rows), u13_TASK, 'independent_ALE_3D_ROI_definition_5_regions')
    return u13_pd.DataFrame(rows)


In [ ]:
import matplotlib.pyplot as u13_plt
import numpy as u13_np
import pandas as u13_pd
import subprocess as u13_subprocess
import matplotlib as u13_matplotlib
u13_matplotlib.use('Agg')
import matplotlib.pyplot as u13_plt
import numpy as u13_np
import pandas as u13_pd
import numpy as u13_np
import pandas as u13_pd
import matplotlib.pyplot as u13_plt
from pathlib import Path as u15_Path
import os as u15_os, hashlib as u15_hashlib, json as u15_json
import numpy as u15_np
import pandas as u15_pd
u15_ROOT = ANALYSIS_ROOT
u15_DATA = u15_Path(u15_os.environ.get('A8_DATA_DIR', NOTEBOOK_PATH.parent)).expanduser().resolve()
u15_RESULTS = u15_Path(u15_os.environ.get('A8_OUTPUT_DIR', u15_ROOT / 'results')).expanduser().resolve()
u15_EXPECTED_N = int(u15_os.environ.get('A8_EXPECTED_N', '106'))
u15_PARAMS = ['stroop_a', 'stroop_v_congruent', 'stroop_v_neutral', 'stroop_v_incongruent', 'stroop_t', 'gonogo_a', 'gonogo_v_go', 'gonogo_v_nogo', 'gonogo_t', 'gonogo_z']
u15_INPUT_FOLDERS = {'MRI_manifest.csv': '7.task_fmri', 'Participants.csv': '1.behavior', 'ROI_masks.csv': '7.task_fmri', 'HDDM_posterior_draws.csv': '2.hddm_fitting', 'HDDM_parameters.csv': '2.hddm_fitting', 'Behavior_data.csv': '1.behavior', 'HDDM_data.csv': '2.hddm_fitting'}
u15_INPUT_FOLDERS.update({'MRI_preprocessing.csv': '7.task_fmri', 'MultiEcho_runs.csv': '7.task_fmri'})
u15_INPUT_FILES = {name: name for name in u15_INPUT_FOLDERS}
u15_INPUT_COLUMNS = {'Participants.csv': ('subject_id', 'age', 'sex', 'education'), 'Behavior_data.csv': ('subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'), 'HDDM_data.csv': ('subj_idx', 'task', 'run', 'trial_id', 'condition', 'stimulus', 'response', 'rt'), 'MRI_manifest.csv': ('subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'), 'ROI_masks.csv': ('roi_id', 'path', 'source_doi', 'independent_of_sample'), 'HDDM_parameters.csv': ('subject_id', *u15_PARAMS), 'HDDM_posterior_draws.csv': ('task', 'chain', 'draw', 'subject_id', 'parameter', 'value')}


In [ ]:
def u15_input_path(name):
    filename = u15_INPUT_FILES.get(name, name)
    p = u15_DATA / filename
    if not p.is_file() and 'A8_DATA_DIR' not in u15_os.environ:
        p = u15_ROOT / u15_INPUT_FOLDERS.get(name, '7.task_fmri') / filename
    if name == 'MRI_manifest.csv' and (u15_os.environ.get('A8_MRI_INPUT_MODE') == 'generated' or not p.is_file()):
        p = u15_RESULTS / '7.task_fmri/tables/MRI_manifest.csv'
    fitted = (u15_RESULTS / '2.hddm_fitting/fit_state.json').is_file() and u15_os.environ.get('A8_HDDM_INPUT_MODE', 'auto') != 'external'
    if name in {'HDDM_parameters.csv', 'HDDM_posterior_draws.csv'} and (fitted or not p.is_file()):
        p = u15_RESULTS / '2.hddm_fitting/tables' / name
    if not p.is_file():
        raise FileNotFoundError(f'Required observed input missing: {p}. Supply the required real input file; no example values are substituted.')
    return p


In [ ]:
def u15_table(name, columns=(), unique=None):
    p = u15_input_path(name)
    d = u15_pd.read_csv(p, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    if d.empty:
        raise ValueError(f'{p.name}: no observed records; a header-only template is not data')
    missing = (set(columns) | set(u15_INPUT_COLUMNS.get(name, ()))) - set(d)
    if missing:
        raise ValueError(f'{p.name}: missing columns {sorted(missing)}')
    if unique and (d[list(unique)].isna().any().any() or d.duplicated(list(unique)).any()):
        raise ValueError(f'{p.name}: missing/duplicate key {unique}')
    for c in ('subject_id', 'subj_idx', 'trial_id'):
        if c in d and (d[c].isna().any() or d[c].str.strip().eq('').any() or d[c].ne(d[c].str.strip()).any()):
            raise ValueError(f'{p.name}: blank or whitespace-padded identifier in {c}')
    for c in ('data_status', 'source_status'):
        if c in d and d[c].astype(str).str.contains('synthetic|simulation|not empirical', case=False, regex=True).any():
            raise ValueError(f'{p.name}: simulated rows cannot enter this workflow')
    return d


In [ ]:
def u15_participants():
    d = u15_table('Participants.csv', ['subject_id', 'age', 'sex', 'education'], ['subject_id'])
    if len(d) != u15_EXPECTED_N:
        raise ValueError(f'Expected {u15_EXPECTED_N} participants, found {len(d)}; configure A8_EXPECTED_N deliberately.')
    for c in ['age', 'sex', 'education']:
        d[c] = u15_pd.to_numeric(d[c], errors='raise')
        if not u15_np.isfinite(d[c]).all():
            raise ValueError(f'Nonfinite {c}')
    if not d.sex.isin([0, 1]).all():
        raise ValueError('sex must use the documented 0/1 coding')
    return d.sort_values('subject_id').reset_index(drop=True)


In [ ]:
def u15_match_cohort(d, name, keys=('subject_id',), complete=True):
    ids = set(u15_participants().subject_id)
    if d.subject_id.isna().any() or not set(d.subject_id) <= ids:
        raise ValueError(f'{name}: unexpected or missing subject_id')
    if complete and set(d.subject_id) != ids:
        raise ValueError(f'{name}: cohort mismatch; do not fill missing participants')
    if d.duplicated(list(keys)).any():
        raise ValueError(f'{name}: duplicate {keys}')
    return d


In [ ]:
def u15_file_ref(value):
    if u15_pd.isna(value) or not str(value).strip():
        raise ValueError('Empty input path')
    p = u15_Path(str(value)).expanduser()
    p = p if p.is_absolute() else u15_DATA / p
    if not p.is_file():
        raise FileNotFoundError(p)
    return p.resolve()


In [ ]:
def u15_imaging_manifest():
    d = u15_table('MRI_manifest.csv', ['subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'])
    d['run'] = d.run.fillna('').str.replace('\\.0$', '', regex=True).str.lstrip('0').replace({'': 'single'})
    u15_match_cohort(d, 'MRI_manifest', ('subject_id', 'task', 'run'))
    if not d.task.isin(['stroop', 'gonogo', 'rest']).all():
        raise ValueError('Invalid MRI task')
    return d


In [ ]:
def u15_imaging_paths(subject, task, run=None):
    run = str(run or 'single').lstrip('0') or 'single'
    d = u15_imaging_manifest()
    g = d[(d.subject_id == subject) & (d.task == task) & (d.run == run)]
    if len(g) != 1:
        raise ValueError(f'Expected one MRI run: {subject}/{task}/{run}, found {len(g)}')
    row = g.iloc[0]
    cols = ['bold', 'mask', 'confounds', 'metadata'] + ([] if task == 'rest' else ['events'])
    return {c: u15_file_ref(row[c]) for c in cols}


In [ ]:
def u15_source_signature():
    document = u15_json.loads(NOTEBOOK_PATH.read_text())
    source = '\n'.join((''.join(cell['source']) for cell in document['cells'] if cell['cell_type'] == 'code'))
    return u15_hashlib.sha256(source.encode()).hexdigest()


In [ ]:
from pathlib import Path as u17__A8Path
from pathlib import Path as u17_Path
import sys as u17_sys
import hashlib as u17_hashlib
import json as u17_json
import os as u17_os
for u17__key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u17_os.environ.setdefault(u17__key, '1')
import nibabel as u17_nib
import numpy as u17_np
import pandas as u17_pd
import nilearn as u17_nilearn
if u17_nilearn.__version__ != '0.13.1':
    raise RuntimeError('Use Nilearn 0.13.1, as specified in the manuscript')
from scipy import stats as u17_stats
from statsmodels.stats.multitest import multipletests as u17_multipletests
import sys as u17_sys
from pathlib import Path as u17_Path
u17_PROJECT = u15_ROOT
u17_RESULTS = u15_RESULTS
u17_DATA = u15_DATA
u17_EXPECTED_N = u15_EXPECTED_N
u17_participants = u15_participants
u17_source_signature = u15_source_signature
u17_MRI = u17_RESULTS / '3.mri_analysis'
u17_TASK = u17_MRI / '2.task_fmri'
u17_REST = u17_MRI / '1.resting_state'
u17_VERSION = 'observed_inputs_v1'
u17_LEGACY = u17_DATA / 'ale_masks'
u17_NEW = u17_TASK / 'other_outputs'
u17_REST_NEW = u17_REST / 'other_outputs'
u17_ROI_CONFIG = u17_MRI / 'ROI_definition_ALE_3D_clusters.json'
u17_ROI_VARIANTS = ('ALE_3D',)
u17_ROIS = ['shared_SMA', 'shared_rAI_FO', 'shared_lAI', 'stroop_lIPS', 'gonogo_rIFG_PMC']
u17_CONTRAST = {'stroop': 'incongruent_minus_congruent', 'gonogo': 'nogo_correct_minus_go_correct'}
u17_N_PERM = 50000
u17_SEED = 20261017


In [ ]:
def u17_subjects():
    return u17_participants().subject_id.tolist()


In [ ]:
def u17_sha(path):
    h = u17_hashlib.sha256()
    with u17_Path(path).open('rb') as f:
        for block in iter(lambda : f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


In [ ]:
def u17_fingerprint(paths, extra=None):
    info = [(str(u17_Path(p).resolve()), u17_Path(p).stat().st_size, u17_Path(p).stat().st_mtime_ns) for p in paths]
    return u17_hashlib.sha256(u17_json.dumps([info, extra, u17_source_signature()], sort_keys=True).encode()).hexdigest()


In [ ]:
def u17_atomic_json(path, data):
    path = u17_Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')

    def native(value):
        if isinstance(value, u17_np.generic):
            return value.item()
        if isinstance(value, u17_np.ndarray):
            return value.tolist()
        raise TypeError(f'Unsupported metadata type: {type(value).__name__}')
    tmp.write_text(u17_json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False, default=native))
    tmp.replace(path)


In [ ]:
def u17_save_table(frame, root, name):
    if '10_activation_tests' in name or name.startswith('functional_connectivity_10_edges_'):
        frame = frame.drop(columns=['p_flip_positive'], errors='ignore')
    if name.startswith('functional_connectivity_10_edges_'):
        frame = frame.drop(columns=['t', 'df', 'p_t'], errors='ignore')
    if 'pattern_similarity_' in name:
        frame = frame.drop(columns=['t', 'df', 'p_t'], errors='ignore')
    dest = root / 'tables'
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f'{name}.xlsx'
    frame.to_excel(path, index=False)
    from openpyxl import load_workbook as load_workbook
    from openpyxl.styles import Font as Font, Border as Border, Side as Side, Alignment as Alignment
    from openpyxl.utils import get_column_letter as get_column_letter
    wb = load_workbook(path)
    ws = wb.active
    rule = Side(style='thin', color='333333')
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = 'A2'
    for (col, key) in enumerate(frame.columns, 1):
        letter = get_column_letter(col)
        ws.column_dimensions[letter].width = min(38, max(13, len(str(key)) + 2))
        cell = ws.cell(1, col)
        cell.font = Font(name='Arial', size=10, bold=True)
        cell.alignment = Alignment(wrap_text=True, vertical='center')
        cell.border = Border(top=rule, bottom=rule)
        for row in range(2, ws.max_row + 1):
            c = ws.cell(row, col)
            c.font = Font(name='Arial', size=10)
            if isinstance(c.value, str):
                c.alignment = Alignment(wrap_text=True, vertical='top')
                visual_length = sum((2 if ord(char) > 255 else 1 for char in c.value))
                lines = max(1, int(u17_np.ceil(visual_length / ws.column_dimensions[letter].width)))
                ws.row_dimensions[row].height = max(ws.row_dimensions[row].height or 15, min(150, lines * 15))
            if isinstance(c.value, float):
                c.number_format = '0.000E+00' if str(key).startswith(('p', 'q')) else '0.000'
            if row == ws.max_row:
                c.border = Border(bottom=rule)
    ws.row_dimensions[1].height = 32
    wb.save(path)


In [ ]:
def u17_fdr(p):
    p = u17_np.asarray(p, float)
    out = u17_np.full(p.shape, u17_np.nan)
    valid = u17_np.isfinite(p)
    if valid.any():
        out[valid] = u17_multipletests(p[valid], method='fdr_bh')[1]
    return out


In [ ]:
def u17_mean_stats(x, seed=u17_SEED, alternative='two-sided'):
    if alternative not in ('two-sided', 'greater'):
        raise ValueError('Unsupported alternative')
    key = 'p_flip' if alternative == 'two-sided' else 'p_flip_positive'
    x = u17_np.asarray(x, float)
    x = x[u17_np.isfinite(x)]
    n = len(x)
    if n < 3:
        return dict(N=n, mean=u17_np.nan, SD=u17_np.nan, t=u17_np.nan, df=n - 1, CI95_low=u17_np.nan, CI95_high=u17_np.nan, **{key: u17_np.nan})
    mu = x.mean()
    rng = u17_np.random.default_rng(seed)
    exceed = 0
    for start in range(0, u17_N_PERM, 1000):
        signs = rng.choice((-1.0, 1.0), size=(min(1000, u17_N_PERM - start), n))
        perm = signs @ x / n
        exceed += u17_np.count_nonzero(abs(perm) >= abs(mu) - 1e-14) if alternative == 'two-sided' else u17_np.count_nonzero(perm >= mu - 1e-14)
    sem = u17_stats.sem(x)
    ci = u17_stats.t.interval(0.95, n - 1, loc=mu, scale=sem)
    return dict(N=n, mean=float(mu), SD=float(x.std(ddof=1)), t=float(mu / sem), df=n - 1, CI95_low=float(ci[0]), CI95_high=float(ci[1]), **{key: (exceed + 1) / (u17_N_PERM + 1)})


In [ ]:
def u17_first_path(sid, task, stat):
    name = f'{sid}_task-{task}_contrast-{u17_CONTRAST[task]}_stat-{stat}.nii.gz'
    fresh = u17_NEW / name
    if not u17_new_first_complete(sid, task):
        raise FileNotFoundError(f'Current first-level GLM is incomplete; fallback to previous input results is prohibited: {sid}/{task}')
    return fresh


In [ ]:
def u17_new_first_complete(sid, task):
    suffix = 'stroop' if task == 'stroop' else 'gonogo_duration500ms'
    qc = u17_NEW / f'{sid}_task-{suffix}_first_level_qc.csv'
    maps = [u17_NEW / f'{sid}_task-{task}_contrast-{u17_CONTRAST[task]}_stat-{stat}.nii.gz' for stat in ('effect', 'variance', 'z')]
    maps.append(u17_NEW / f'{sid}_task-{task}_mask-analysis.nii.gz')
    return qc.exists() and all((p.exists() for p in maps))


In [ ]:
def u17_qc_path(sid, task):
    suffix = 'stroop' if task == 'stroop' else 'gonogo_duration500ms'
    name = f'{sid}_task-{suffix}_first_level_qc.csv'
    fresh = u17_NEW / name
    if not u17_new_first_complete(sid, task):
        raise FileNotFoundError(f'Current first-level GLM is incomplete: {sid}/{task}')
    return fresh


In [ ]:
def u17_t_map(sid, task):
    e = u17_nib.load(u17_first_path(sid, task, 'effect'))
    v = u17_nib.load(u17_first_path(sid, task, 'variance'))
    if e.shape != v.shape or not u17_np.allclose(e.affine, v.affine):
        raise ValueError('Effect and variance image spaces do not match')
    (ed, vd) = (u17_np.asarray(e.dataobj, float), u17_np.asarray(v.dataobj, float))
    valid = u17_np.isfinite(ed) & u17_np.isfinite(vd) & (vd > 0)
    t = u17_np.divide(ed, u17_np.sqrt(u17_np.maximum(vd, 0)), out=u17_np.zeros_like(ed), where=valid)
    qc = u17_pd.read_csv(u17_qc_path(sid, task), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    dof = int((qc.n_timepoints - qc.design_rank).sum())
    return (t, valid, e, dof)


In [ ]:
def u17_roi_masks(definition='ALE_3D'):
    if definition != 'ALE_3D':
        raise ValueError('The specified analysis uses independent three-dimensional ALE cluster masks only')
    cfg = u17_json.loads(u17_ROI_CONFIG.read_text())
    manifest = u17_NEW / 'definition_and_provenance.json'
    if not manifest.exists():
        raise FileNotFoundError('Provide and validate independent ALE ROI inputs, or run the ROI definition step in this notebook')
    meta = u17_json.loads(manifest.read_text())
    if meta['config_sha256'] != u17_sha(u17_ROI_CONFIG):
        raise ValueError('ROI definition changed; regenerate ROI masks')
    ref = u17_nib.load(u17_first_path(u17_subjects()[0], 'stroop', 'effect'))
    masks = {}
    for roi in u17_ROIS:
        path = u17_NEW / f'{roi}_ALE_3D.nii.gz'
        if u17_sha(path) != meta['masks'][roi]['sha256']:
            raise ValueError('ALE mask validation failed')
        img = u17_nib.load(path)
        data = u17_np.asarray(img.dataobj)
        if img.shape != ref.shape or not u17_np.allclose(img.affine, ref.affine):
            raise ValueError('ALE ROI and subject statistic image spaces do not match')
        if not u17_np.isin(data, [0, 1]).all() or not data.any():
            raise ValueError('ALE mask must be nonempty and binary')
        masks[roi] = data > 0
    return (masks, ref)


In [ ]:
u18_imaging_paths = u15_imaging_paths
u18_imaging_manifest = u15_imaging_manifest
u18_RESULTS = u15_RESULTS
import hashlib as u18_hashlib, json as u18_json
u18_MANIFEST = u18_RESULTS / '3.mri_analysis/2.task_fmri/other_outputs/task_input_provenance.json'


In [ ]:
def u18_freeze():
    records = []
    for row in u18_imaging_manifest().itertuples():
        p = u18_imaging_paths(row.subject_id, row.task, None if row.run == 'single' else row.run)
        records.append({'subject_id': row.subject_id, 'task': row.task, 'run': row.run, 'files': {k: {'path': str(v), 'sha256': u18_hashlib.sha256(v.read_bytes()).hexdigest()} for (k, v) in p.items()}})
    u18_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    u18_MANIFEST.write_text(u18_json.dumps({'status': 'input_files_verified', 'runs': records}, ensure_ascii=False, indent=2))
    return records


In [ ]:
u20_subjects = u17_subjects
u20_atomic_json = u17_atomic_json
u20_NEW = u17_NEW


In [ ]:
def u20_fit_job(job):
    (sid, task) = job
    if u12_glm_process_subject(sid, tasks=[task]):
        raise RuntimeError(f'First-level GLM failed: {sid}/{task}')
    return job


In [ ]:
def u20_first_levels():
    MANIFEST = u18_MANIFEST
    freeze = u18_freeze
    freeze()
    from joblib import Parallel as Parallel, delayed as delayed
    jobs = [(sid, task) for sid in u20_subjects() for task in ('stroop', 'gonogo')]
    for (index, job) in enumerate(Parallel(n_jobs=3, backend='loky')((delayed(u20_fit_job)(job) for job in jobs)), 1):
        u20_atomic_json(u20_NEW / 'first_level_run_status.json', dict(status='running', completed=index, total=len(jobs), last=job))
    u20_atomic_json(u20_NEW / 'first_level_run_status.json', dict(status='complete', completed=len(jobs), total=len(jobs)))


In [ ]:
u12_ROOT = u15_ROOT
u12_DATA = u15_DATA
u12_RESULTS = u15_RESULTS
u12_PROJECT = u15_ROOT
u12_DATA = u15_DATA
u12_RESULTS = u15_RESULTS
u12_participants = u15_participants
u12_input_path = u15_input_path
u12_imaging_paths = u15_imaging_paths
u12_matplotlib.use('Agg')
u12_source_signature = u15_source_signature
u12__A8Path = u17__A8Path
u12_Path = u17_Path
u12_sys = u17_sys
u12_hashlib = u17_hashlib
u12_json = u17_json
u12_os = u17_os
u12_nib = u17_nib
u12_np = u17_np
u12_pd = u17_pd
u12_nilearn = u17_nilearn
u12_stats = u17_stats
u12_multipletests = u17_multipletests
u12_PROJECT = u17_PROJECT
u12_RESULTS = u17_RESULTS
u12_DATA = u17_DATA
u12_EXPECTED_N = u17_EXPECTED_N
u12_participants = u17_participants
u12_source_signature = u17_source_signature
u12_MRI = u17_MRI
u12_TASK = u17_TASK
u12_REST = u17_REST
u12_VERSION = u17_VERSION
u12_LEGACY = u17_LEGACY
u12_NEW = u17_NEW
u12_REST_NEW = u17_REST_NEW
u12_ROI_CONFIG = u17_ROI_CONFIG
u12_ROI_VARIANTS = u17_ROI_VARIANTS
u12_ROIS = u17_ROIS
u12_CONTRAST = u17_CONTRAST
u12_N_PERM = u17_N_PERM
u12_SEED = u17_SEED
u12_subjects = u17_subjects
u12_sha = u17_sha
u12_fingerprint = u17_fingerprint
u12_atomic_json = u17_atomic_json
u12_save_table = u17_save_table
u12_fdr = u17_fdr
u12_mean_stats = u17_mean_stats
u12_first_path = u17_first_path
u12_new_first_complete = u17_new_first_complete
u12_qc_path = u17_qc_path
u12_t_map = u17_t_map
u12_roi_masks = u17_roi_masks
u12__A8Path = u17__A8Path
u12_Path = u17_Path
u12_sys = u17_sys
u12_hashlib = u17_hashlib
u12_json = u17_json
u12_os = u17_os
u12_nib = u17_nib
u12_np = u17_np
u12_pd = u17_pd
u12_nilearn = u17_nilearn
u12_stats = u17_stats
u12_multipletests = u17_multipletests
u12_PROJECT = u17_PROJECT
u12_RESULTS = u17_RESULTS
u12_DATA = u17_DATA
u12_EXPECTED_N = u17_EXPECTED_N
u12_participants = u17_participants
u12_source_signature = u17_source_signature
u12_MRI = u17_MRI
u12_TASK = u17_TASK
u12_REST = u17_REST
u12_VERSION = u17_VERSION
u12_LEGACY = u17_LEGACY
u12_NEW = u17_NEW
u12_REST_NEW = u17_REST_NEW
u12_ROI_CONFIG = u17_ROI_CONFIG
u12_ROI_VARIANTS = u17_ROI_VARIANTS
u12_ROIS = u17_ROIS
u12_CONTRAST = u17_CONTRAST
u12_N_PERM = u17_N_PERM
u12_SEED = u17_SEED
u12_subjects = u17_subjects
u12_sha = u17_sha
u12_fingerprint = u17_fingerprint
u12_atomic_json = u17_atomic_json
u12_save_table = u17_save_table
u12_fdr = u17_fdr
u12_mean_stats = u17_mean_stats
u12_first_path = u17_first_path
u12_new_first_complete = u17_new_first_complete
u12_qc_path = u17_qc_path
u12_t_map = u17_t_map
u12_roi_masks = u17_roi_masks
u12_plt.rcParams.update({'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold', 'mathtext.bfit': 'Arial:italic:bold', 'mathtext.sf': 'Arial'})
u12_ROIS = u17_ROIS
u12_CONTRAST = u17_CONTRAST
u12_MRI = u17_MRI
u12_matplotlib.use('Agg')
u12_MRI = u17_MRI
u12_REST = u17_REST
u12_LEGACY = u17_LEGACY
u12_NEW = u17_NEW
u12_ROIS = u17_ROIS
u12_sha = u17_sha
u12_input_path = u15_input_path
u12_source_signature = u15_source_signature


In [ ]:
@u12_lru_cache(maxsize=1)
def u12_brain_meshes():
    return u12_datasets.fetch_surf_fsaverage(mesh='fsaverage6')


In [ ]:
@u12_lru_cache(maxsize=1)
def u12_brain_background():
    return u12_nib.as_closest_canonical(u12_nib.load(u12_input_path('Template_T1w.nii.gz')))


In [ ]:
@u12_lru_cache(maxsize=1)
def u12_brain_background_mask():
    path = u12_Path.home() / '.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_mask.nii.gz'
    if not path.is_file():
        from templateflow.api import get as get
        path = u12_Path(get('MNI152NLin2009cAsym', resolution=2, desc='brain', suffix='mask', extension='.nii.gz'))
    mask = u12_nib.as_closest_canonical(u12_nib.load(path))
    bg = u12_brain_background()
    assert mask.shape == bg.shape and u12_np.allclose(mask.affine, bg.affine)
    return u12_np.asarray(mask.dataobj) > 0


In [ ]:
@u12_lru_cache(maxsize=8)
def u12_brain_cortical_luminance(mesh_path, sulc_path, hemi, view):
    (xyz, faces) = u12_surface.load_surf_mesh(mesh_path)
    xyz = u12_np.asarray(xyz, dtype=float)
    faces = u12_np.asarray(faces)
    points = xyz[faces]
    area_normal = u12_np.cross(points[:, 1] - points[:, 0], points[:, 2] - points[:, 0])
    if u12_np.median(u12_np.sum(area_normal * (points.mean(axis=1) - xyz.mean(axis=0)), axis=1)) < 0:
        area_normal = -area_normal
    vertex_normal = u12_np.zeros_like(xyz)
    for corner in range(3):
        u12_np.add.at(vertex_normal, faces[:, corner], area_normal)
    vertex_normal /= u12_np.maximum(u12_np.linalg.norm(vertex_normal, axis=1, keepdims=True), 1e-12)
    normal = vertex_normal[faces].mean(axis=1)
    normal /= u12_np.maximum(u12_np.linalg.norm(normal, axis=1, keepdims=True), 1e-12)
    side = -1 if (hemi == 'left') == (view == 'lateral') else 1
    key = u12_np.array([side * 0.8, -0.45, 0.75])
    key /= u12_np.linalg.norm(key)
    fill = u12_np.array([side * 0.6, 0.65, 0.2])
    fill /= u12_np.linalg.norm(fill)
    illumination = 0.55 + 0.38 * u12_np.maximum(normal @ key, 0) + 0.07 * u12_np.maximum(normal @ fill, 0)
    sulc = u12_np.asarray(u12_surface.load_surf_data(sulc_path))
    (lo, hi) = u12_np.percentile(sulc, [2, 98])
    curvature = u12_np.clip((sulc - lo) / (hi - lo), 0, 1)[faces].mean(axis=1)
    luminance = (0.94 - 0.18 * curvature) * illumination
    assert u12_np.isfinite(luminance).all() and u12_np.all((luminance >= 0) & (luminance <= 1))
    return (faces, luminance)


In [ ]:
@u12_lru_cache(maxsize=1)
def u12_brain_inverse_transform():
    p = u12_input_path('MNI2009c_to_MNI152_display_transform.h5')
    if not p.is_file() or p.stat().st_size < 1000:
        raise FileNotFoundError('TemplateFlow MNI2009c-to-MNI152 display transform is required')
    return p


In [ ]:
u12_ROIS = u17_ROIS
u12_roi_masks = u17_roi_masks
u12__A8Path = u17__A8Path
u12_Path = u17_Path
u12_sys = u17_sys
u12_hashlib = u17_hashlib
u12_json = u17_json
u12_os = u17_os
u12_nib = u17_nib
u12_np = u17_np
u12_pd = u17_pd
u12_nilearn = u17_nilearn
u12_stats = u17_stats
u12_multipletests = u17_multipletests
u12_PROJECT = u17_PROJECT
u12_RESULTS = u17_RESULTS
u12_DATA = u17_DATA
u12_EXPECTED_N = u17_EXPECTED_N
u12_participants = u17_participants
u12_source_signature = u17_source_signature
u12_MRI = u17_MRI
u12_TASK = u17_TASK
u12_REST = u17_REST
u12_VERSION = u17_VERSION
u12_LEGACY = u17_LEGACY
u12_NEW = u17_NEW
u12_REST_NEW = u17_REST_NEW
u12_ROI_CONFIG = u17_ROI_CONFIG
u12_ROI_VARIANTS = u17_ROI_VARIANTS
u12_ROIS = u17_ROIS
u12_CONTRAST = u17_CONTRAST
u12_N_PERM = u17_N_PERM
u12_SEED = u17_SEED
u12_subjects = u17_subjects
u12_sha = u17_sha
u12_fingerprint = u17_fingerprint
u12_atomic_json = u17_atomic_json
u12_save_table = u17_save_table
u12_fdr = u17_fdr
u12_mean_stats = u17_mean_stats
u12_first_path = u17_first_path
u12_new_first_complete = u17_new_first_complete
u12_qc_path = u17_qc_path
u12_t_map = u17_t_map
u12_roi_masks = u17_roi_masks
u13_Path = u17_Path
u13_json = u17_json
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_TASK = u17_TASK
u13_NEW = u17_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROIS = u17_ROIS
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_first_path = u17_first_path
u13_Path = u17_Path
u13_json = u17_json
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_TASK = u17_TASK
u13_NEW = u17_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROIS = u17_ROIS
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_first_path = u17_first_path
u13_Path = u17_Path
u13_json = u17_json
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_TASK = u17_TASK
u13_NEW = u17_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROIS = u17_ROIS
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_first_path = u17_first_path
u13_plt.rcParams.update({'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold', 'mathtext.bfit': 'Arial:italic:bold', 'mathtext.sf': 'Arial'})
u13_ROIS = u17_ROIS
u13_Path = u17_Path
u13_json = u17_json
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_TASK = u17_TASK
u13_NEW = u17_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROIS = u17_ROIS
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_first_path = u17_first_path
u13_plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.7, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'xtick.labelsize': 6, 'ytick.labelsize': 7, 'legend.frameon': False})
u13_TASK = u17_TASK
u13_ROIS = u17_ROIS
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u12_sys.dont_write_bytecode = True
for u12_key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u12_os.environ.setdefault(u12_key, '1')
u12_glm_ANALYSIS = u12_RESULTS / '3.mri_analysis/2.task_fmri'
u12_glm_OUT_ROOT = u12_glm_ANALYSIS / 'other_outputs'
u12_glm_QC_ROOT = u12_glm_OUT_ROOT
u12_sys.dont_write_bytecode = True
for u12_key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u12_os.environ.setdefault(u12_key, '1')
u12_group_SYSTEM_FONT = u12_Path('/System/Library/Fonts/Supplemental/Arial.ttf')
if u12_group_SYSTEM_FONT.exists():
    u12_font_manager.fontManager.addfont(u12_group_SYSTEM_FONT)
    u12_plt.rcParams['font.family'] = 'Arial'
u12_ANALYSIS = u12_glm_ANALYSIS
u12_OUT_ROOT = u12_glm_OUT_ROOT
u12_group_GROUP_ROOT = u12_ANALYSIS / 'other_outputs'
u12_sys.dont_write_bytecode = True
u12_style_ROOT = u12_MRI
u12_sys.dont_write_bytecode = True
u12_brain_CACHE = u12_MRI / 'display_cache'
u12_brain_CACHE.mkdir(exist_ok=True, parents=True)
u12_sys.dont_write_bytecode = True
u12_sys.dont_write_bytecode = True


In [ ]:
def a8_display_three_line(value, *args, **kwargs):
    import pandas as pd
    from IPython.display import display, HTML
    if isinstance(value, pd.DataFrame):
        html = value.to_html(index=False, border=0, escape=True)
        html = html.replace('<table ', '<table style="border-collapse:collapse;border-top:2px solid black;border-bottom:2px solid black;" ', 1)
        html = html.replace('<th>', '<th style="border:0;border-bottom:1px solid black;padding:7px 16px;text-align:center;">')
        html = html.replace('<td>', '<td style="border:0;padding:7px 16px;text-align:center;">')
        return display(HTML(html), *args, **kwargs)
    return display(value, *args, **kwargs)
u12_display = a8_display_three_line


In [ ]:
u12_first_levels = u20_first_levels
u12_first_levels()
u12_qc = u12_alignment_audit_first()
u12_display(u12_qc.groupby('task')[['dof', 'mean_fd_mm']].agg(['min', 'mean', 'max']))


In [ ]:
u12_alignment_group()
u12_define = u13_roi_define
u12_alignment_overlap()
u12_define()
u12_brains()
u12_qc_summary()
u12_max_t()
u12_display(u12_pd.read_excel(u12_TASK / 'tables/group_level_permutation_cluster_FWE.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}))
u12_display(u12_Image(filename=str(u12_TASK / 'image/Figure_S1_representative_design_matrix_n106.png'), width=600))
u12_display(u12_Image(filename=str(u12_TASK / 'image/Figure_S1_head_motion_distribution_n106.png'), width=600))
u12_display(u12_Image(filename=str(u12_TASK / 'image/Figure_S1_Stroop_standard_space_brain_coverage_n106.png'), width=1000))
u12_display(u12_Image(filename=str(u12_TASK / 'image/Figure_S1_GoNoGo_standard_space_brain_coverage_n106.png'), width=1000))
u12_display(u12_Image(filename=str(u12_TASK / 'image/Figure_S4_Stroop_voxelwise_maxT_FWE_n106.png'), width=1000))
u12_display(u12_Image(filename=str(u12_TASK / 'image/Figure_S4_GoNoGo_voxelwise_maxT_FWE_n106.png'), width=1000))
